<a href="https://colab.research.google.com/github/Mac-Tapia/CityLearn/blob/citylearn-v3-madrl/examples/madrl_citylearn_v3_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MULTI-AGENTE DE APRENDIZAJE POR REFUERZO PROFUNDO PARA GESTIÓN COORDINADA DE FLEXIBILIDAD ENERGÉTICA, EMISIONES DE CARBONO Y EFICIENCIA ECONÓMICA EN COMUNIDADES INTELIGENTES

## Tutorial CityLearn v3 MADRL sobre CityLearn v2

Este notebook sigue la estructura pedagogica del tutorial original de CityLearn: contexto, datos, preprocesamiento, entorno, KPIs, visualizacion, control, entrenamiento, evaluacion, ajuste y siguientes pasos. La diferencia es que aqui el controlador ya no es un unico agente RL, sino un sistema **MADRL colaborativo** con **Dec-POMDP**, **CTDE** y cuatro backends oficiales: **HAPPO**, **MASAC**, **MATD3** y **MAAC**.

La idea central del proyecto es conservar **CityLearn v2** como simulador, dataset, fisica y fuente oficial de KPIs, agregando una capa **CityLearn v3** para entrenamiento multiagente profundo y evaluacion multiobjetivo.


# Glossary

- **CityLearn v2**: simulador base usado para edificios, baterias, PV, EVs, tarifas, intensidad de carbono y KPIs `evaluate_v2`.
- **CityLearn v3 MADRL**: capa experimental de este proyecto que adapta CityLearn v2 a Dec-POMDP, CTDE, backends MADRL oficiales y reportes por ejes.
- **MADRL**: aprendizaje por refuerzo profundo multiagente.
- **Dec-POMDP**: juego de Markov parcialmente observable descentralizado; cada edificio observa localmente y actua localmente.
- **CTDE**: entrenamiento centralizado y ejecucion descentralizada; el critico puede usar estado global durante entrenamiento, pero cada actor ejecuta con informacion local.
- **HAPPO**: actor-critic multiagente de HARL, usado con critico centralizado.
- **MASAC**: variante multiagente de Soft Actor-Critic usada sobre un estado global estilo SMAC.
- **MATD3**: TD3 multiagente con critico centralizado y actores continuos por agente.
- **MAAC**: Actor-Attention-Critic multiagente con critico de atencion.
- **OE1**: flexibilidad energetica.
- **OE2**: emisiones de CO2.
- **OE3**: costos energeticos.
- **Baseline CityLearn v2**: referencia usada por CityLearn para calcular ratios, deltas y comparaciones de KPIs.


<a name="overview"></a>

# Overview

El tutorial original de CityLearn enseña como pasar de datos y reglas de control a agentes RL que modifican acciones de almacenamiento. Este notebook conserva esa ruta de aprendizaje, pero cambia el foco hacia comunidades inteligentes donde cada edificio es un agente coordinado.

El flujo de trabajo sera:

1. Revisar el objetivo cientifico y los ejes de evaluacion.
2. Cargar el dataset CityLearn v2 con 17 edificios + EV.
3. Inspeccionar clima, precios, carbono y archivos de edificios.
4. Construir el entorno Dec-POMDP de CityLearn v3.
5. Validar observaciones locales, acciones locales y estado global CTDE.
6. Evaluar KPIs CityLearn v2 y KPIs del proyecto por OE1/OE2/OE3.
7. Revisar los cuatro backends MADRL oficiales.
8. Ejecutar entrenamientos cortos o lanzar entrenamiento oficial.
9. Analizar `results.json`, `timeseries.csv`, `trace.csv`, checkpoints, figuras y tablas.
10. Comparar algoritmos contra la linea base CityLearn v2.


## Arquitectura y flujo renderizables del proyecto

Los cambios recientes del proyecto quedaron documentados en un plano y en un Markdown maestro listo para renderizar con Mermaid:

- `docs/ARQUITECTURA_Y_FLUJO_TRABAJO_CITYLEARN_V3_MADRL.md`
- `docs/PLANO_REAL_IMPLEMENTADO_CITYLEARN_V3_MADRL.pdf`
- `docs/PLANO_INTEGRADO_CITYLEARN_V3_MADRL.pdf`

El flujo real implementado es: dataset oficial -> CityLearn v2 -> capa CityLearn v3 -> adaptador comun Dec-POMDP/CTDE -> cuatro scripts MADRL -> launcher `-Scenario ALL` -> artefactos por eje -> benchmark CityLearn v2 -> comparador v2 vs v3.


## Contributions

Este proyecto aporta una integracion reproducible para estudiar control coordinado en comunidades de edificios:

- Mantiene CityLearn v2 como fuente oficial de datos, dinamica fisica y KPIs.
- Expone cada edificio como agente descentralizado.
- Incluye EVs dentro de los espacios de accion/observacion de los edificios.
- Permite CTDE con estado global durante entrenamiento y ejecucion local por edificio.
- Integra cuatro backends MADRL oficiales sin implementar algoritmos dentro de `citylearn.agents`.
- Genera artefactos tecnicos comparables: checkpoints, resultados JSON, series temporales, trazas por agente, figuras y tablas.
- Ordena la evaluacion en tres ejes: flexibilidad, CO2 y costos.


## Learning Outcomes

Al finalizar este notebook deberias poder:

- Explicar por que CityLearn v3 sigue usando CityLearn v2 como entorno de entrenamiento.
- Identificar agentes, observaciones, acciones y estado global en un Dec-POMDP.
- Distinguir KPIs CityLearn v2 de los ejes de evaluacion del proyecto.
- Ejecutar validaciones de estructura antes de entrenar.
- Lanzar entrenamientos cortos para HAPPO, MASAC, MATD3 y MAAC.
- Leer los artefactos generados por cada MADRL.
- Interpretar graficas de convergencia, exploracion, eficiencia, recompensas, returns y comparacion con baseline.


<a name="climate-impact"></a>

# Climate Impact

Las comunidades inteligentes pueden desplazar cargas, usar almacenamiento y coordinar EVs para reducir importaciones en horas criticas. Esta coordinacion tiene tres impactos medibles:

- **Flexibilidad energetica**: reducir picos, rampas y dependencia de importacion desde red.
- **Emisiones de CO2**: evitar consumo en horas con alta intensidad de carbono.
- **Eficiencia economica**: reducir costo total, aprovechar tarifas dinamicas y limitar demanda pico.

El aporte MADRL consiste en aprender politicas coordinadas para muchos edificios sin exigir que cada edificio observe todo el distrito en ejecucion.


## Diagnóstico de la realidad

El problema energético de edificios no es marginal. UNEP y GlobalABC reportan que, en 2022, edificios y construcción concentraron cerca del 34% de la demanda energética global y 37% de las emisiones de CO2 relacionadas con energía y procesos, además de una brecha creciente frente a la trayectoria necesaria de descarbonización (United Nations Environment Programme & Global Alliance for Buildings and Construction, 2024). La IEA estima que la operación de edificios representa alrededor del 30% del consumo final de energía y 26% de emisiones energéticas globales; también advierte que el sector debe acelerar eficiencia, electrificación, resiliencia y reducción de emisiones para alinearse con el escenario Net Zero (International Energy Agency, 2023).

En comunidades urbanas, la penetración de PV, baterías, bombas de calor, vehículos eléctricos y tarifas dinámicas aumenta la flexibilidad disponible, pero también incrementa la complejidad de coordinación. CityLearn fue propuesto precisamente para estandarizar investigación en RL/MARL para respuesta de demanda y gestión energética urbana, en un contexto donde la integración de renovables, almacenamiento y EVs introduce nuevos desafíos operativos para la red (Vázquez-Canteli et al., 2020). CityLearn v2 amplía ese marco hacia comunidades grid-interactive, resilientes, ocupante-céntricas y carbon-aware con DERs, V2G y confort térmico (Nweye et al., 2025).


## Descripción problemática

La problemática de tesis se puede resumir así: una comunidad con 17 edificios y EVs dispone de recursos flexibles, pero las decisiones locales no coordinadas pueden aumentar picos, rampas, importaciones en horas de alta intensidad de carbono y costos bajo tarifas dinámicas. Un controlador centralizado puro puede explotar información global, pero escala mal, reduce privacidad y no representa adecuadamente la ejecución real de edificios heterogéneos. Un controlador independiente por edificio preserva descentralización, pero puede sufrir no estacionariedad, pobre asignación de crédito y decisiones incompatibles con el objetivo distrital.

Por ello se adopta un **Dec-POMDP colaborativo con CTDE**: durante entrenamiento se permite usar estado global para estabilizar críticos y aprendizaje; durante ejecución, cada edificio actúa con su observación local. Esta formulación sigue la motivación clásica de Dec-POMDP para control descentralizado bajo incertidumbre (Bernstein et al., 2002) y la tradición MARL moderna de entrenamiento centralizado con políticas descentralizadas (Lowe et al., 2017; Rashid et al., 2018).

La hipótesis operativa del proyecto es que los cuatro MADRL oficiales pueden aprender políticas colaborativas que mejoren, respecto a la línea base CityLearn v2, uno o más de los ejes: **OE1 flexibilidad energética**, **OE2 emisiones de CO2** y **OE3 eficiencia económica**. La comparación final no se debe basar solo en reward: debe usar KPIs CityLearn v2, series técnicas, trazas por agente, checkpoints y figuras por eje.


<a name="target-audience"></a>

# Target Audience

Este notebook esta pensado para investigadores en energia e IA, estudiantes de RL/MARL, usuarios de CityLearn que necesitan reproducir experimentos con multiples edificios y EV, y evaluadores de tesis que necesitan ver claramente datos, algoritmos, KPIs y artefactos.


<a name="prereqs"></a>

# Prerequisites

Se recomienda tener Python 3.9, el entorno `.venv39-citylearn-v3`, PyTorch con CUDA si se va a entrenar en GPU, repositorios externos bajo `external/`, y conocimientos basicos de RL, actor-critic, SAC/TD3/PPO y evaluacion energetica.

Los entrenamientos completos pueden tardar bastante. Las celdas de entrenamiento incluyen banderas para evitar ejecutar procesos largos por accidente.


<a name="background"></a>

# Background

## Grid-Interactive Efficient Buildings and Energy Flexibility

Los edificios interactivos con la red pueden modificar su consumo neto usando almacenamiento electrico, almacenamiento termico, PV, EVs y control de cargas. La flexibilidad se observa en la forma de la curva agregada: picos, rampas, factor de carga, autoconsumo y exportacion.

## Carbon-Aware District Operation

Cuando existe una serie de intensidad de carbono, la politica puede aprender a desplazar importaciones hacia horas menos intensivas en CO2. Por eso el segundo eje no se trata como metrica secundaria sino como objetivo completo.

## Economic Efficiency

La eficiencia economica combina costo de energia, precios dinamicos, reduccion de picos y respuesta a senales tarifarias. Una politica puede reducir costo sin necesariamente reducir emisiones; por eso los tres ejes se reportan por separado.


## Marco teórico y estado del arte relacionado

### CityLearn y comunidades grid-interactive

CityLearn nace como entorno estandarizado para investigar RL/MARL en respuesta de demanda y gestión energética urbana, buscando facilitar comparación y replicabilidad entre algoritmos (Vázquez-Canteli et al., 2020). CityLearn v2 amplía el alcance hacia comunidades con DERs, EV/V2G, resiliencia, confort y control carbon-aware, lo que lo hace adecuado para evaluar flexibilidad, emisiones y costos en un mismo simulador (Nweye et al., 2025).

### Dec-POMDP, CTDE y coordinación multiagente

El Dec-POMDP formaliza decisiones secuenciales descentralizadas con observabilidad parcial, donde cada agente dispone de información local y el equipo comparte un objetivo global (Bernstein et al., 2002). En MARL profundo, CTDE se usa para reducir no estacionariedad durante entrenamiento sin exigir información global en ejecución. MADDPG introdujo un esquema actor-critic con políticas locales y críticos que pueden observar acciones/observaciones de otros agentes (Lowe et al., 2017). QMIX mostró otra ruta CTDE: aprender un valor conjunto centralizado factorizable en utilidades por agente para ejecución descentralizada (Rashid et al., 2018).

### MADRL para energía, demanda flexible y EVs

La literatura reciente en smart grids y edificios muestra que MARL es pertinente cuando existen muchos recursos distribuidos, información local, privacidad, tarifas dinámicas y necesidad de coordinación. En tesis de maestría, González Rotger (2021) aplicó MARL a HVAC en BEMS y reportó trade-offs entre energía, confort y calidad de aire. Fonseca (2023) estudió integración de activos flexibles, EVs, V2G y comunidades energéticas con MADRL multiobjetivo. Dong (2022) combinó predicción de picos y MARL para gestión de DERs en smart grids con entrenamiento centralizado y ejecución distribuida. En tesis doctoral, Almannouny (2025) abordó pricing dinámico y respuesta de demanda integrada con DRL para sistemas multi-energía.

### Relación con los tres ejes de este proyecto

- **OE1 flexibilidad energética**: se conecta con reducción de picos/rampas, autoconsumo, baterías, EV/V2G y balance comunitario.
- **OE2 emisiones de CO2**: se conecta con control carbon-aware e importaciones en horas de alta intensidad de carbono.
- **OE3 eficiencia económica**: se conecta con precios dinámicos, costos de electricidad, respuesta de demanda y reducción de demanda pico.

El marco teórico justifica que el proyecto no trate estos ejes como métricas aisladas. Son objetivos parcialmente conflictivos, por lo que deben reportarse por separado y compararse con baseline mediante KPIs CityLearn v2.


### KPIs por eje de investigación

Cada eje tiene un conjunto explícito de KPIs. Los ejes son las **métricas científicas del proyecto**; los KPIs son las variables CityLearn v2/v3 usadas para medir cada eje contra la línea base.

| Eje | Objetivo | KPIs incluidos |
|---|---|---|
| **OE1 Flexibilidad energética** | Aumentar desplazamiento de carga, aprovechamiento de baterías, EVs/V2G, PV, autoconsumo e intercambio comunitario. | `grid_import`, `grid_import_control`, `grid_import_baseline`, `grid_import_delta`, `zero_net_energy`, `net_exchange_control`, `net_exchange_baseline`, `net_exchange_delta`, `grid_export_ratio`, `grid_export_control`, `grid_export_baseline`, `grid_export_delta`, `peak_average`, `ramping_average`, `one_minus_load_factor_average`, `pv_generation_total`, `pv_generation_daily_average`, `pv_export_total`, `pv_export_daily_average`, `pv_self_consumption_ratio`, `community_local_traded_total`, `community_local_traded_daily_average`, `community_import_share`, `battery_charge_total`, `battery_discharge_total`, `battery_throughput_total`, `battery_equivalent_full_cycles`, `battery_capacity_fade_ratio`, `ev_departure_count`, `ev_departure_met_count`, `ev_departure_within_tolerance_count`, `ev_departure_success_rate`, `ev_departure_within_tolerance_rate`, `ev_departure_soc_deficit_mean`, `ev_charge_total`, `ev_v2g_export_total`. |
| **OE2 Emisiones de CO2** | Reducir la huella ambiental del distrito y evitar importaciones en horas de alta intensidad de carbono. | `carbon_emissions`, `carbon_emissions_control`, `carbon_emissions_baseline`, `carbon_emissions_delta`, `carbon_emissions_daily_average_control`, `carbon_emissions_daily_average_baseline`, `carbon_emissions_daily_average_delta`. |
| **OE3 Costos energéticos** | Optimizar gasto energético, reducir picos con efecto económico y aprovechar tarifas dinámicas. | `electricity_cost`, `electricity_cost_control`, `electricity_cost_baseline`, `electricity_cost_delta`, `electricity_cost_daily_average_control`, `electricity_cost_daily_average_baseline`, `electricity_cost_daily_average_delta`, `cost_peak_average`, `cost_ramping_average`, `cost_one_minus_load_factor_average`, `price_signal_deviation`. |

`price_signal_deviation` se mantiene como KPI derivado del proyecto porque no es una salida nativa de `evaluate_v2` en este código; se calcula desde importación neta distrital y `electricity_pricing`. Todos los demás KPIs provienen de CityLearn v2 o de agregaciones trazables sobre sus series.


## Control Theories for Smart Communities

En el tutorial original, el usuario pasa de reglas RBC a Q-learning y SAC. En este proyecto el salto conceptual es hacia control multiagente:

- **RBC**: reglas fijas, utiles como linea base y diagnostico.
- **RL centralizado**: un agente decide todas las acciones; puede escalar mal con muchos edificios.
- **MARL/MADRL descentralizado**: cada edificio decide su accion.
- **CTDE**: durante entrenamiento se permite informacion global para estabilizar el aprendizaje; en ejecucion cada actor usa observacion local.


## Reinforcement Learning and MADRL for CityLearn

Cada paso del entorno entrega observaciones locales por edificio, acciones continuas por edificio y recompensas. La formulacion Dec-POMDP usada aqui es:

- agentes: `Building_1`, ..., `Building_17`;
- observacion local: variables CityLearn v2 habilitadas para cada edificio;
- accion local: almacenamiento, EV y otros actuadores disponibles del edificio;
- estado global CTDE: concatenacion de observaciones locales;
- recompensa v3: `CityLearnV3MADRLRewardFunction`, con pesos por eje y perfil por algoritmo;
- agregacion colaborativa Dec-POMDP: `team_mean` por defecto despues de calcular la recompensa v3;
- evaluacion: KPIs CityLearn v2 y reporte CityLearn v3 por ejes.


### Reward Function v3: pesos por eje y perfil por MADRL

La recompensa usada en los entrenamientos CityLearn v3 no usa los pesos base heredados de `MARL` como criterio principal. Los scripts `train_citylearn_v3_*.py` fuerzan `CityLearnV3MADRLRewardFunction`, que combina:

- pesos multiobjetivo por escenario `E1/E2/E3`;
- multiplicadores especificos por MADRL;
- componente EV/V2G separado para SoC, restricciones de carga, autoconsumo y uso de excedentes;
- mezcla local/equipo mediante `team_reward_ratio`.

Esto separa claramente reward de entrenamiento, agregacion colaborativa Dec-POMDP y KPIs CityLearn v2 de evaluacion final.


In [ ]:
# Importa dependencias necesarias para esta seccion.
import pandas as pd

# Importa dependencias de reward CityLearn v3 MADRL.
from citylearn.reward_function import (
    # Ejecuta una instruccion necesaria para esta celda.
    CITYLEARN_V3_AXIS_REWARD_WEIGHTS,
    # Ejecuta una instruccion necesaria para esta celda.
    CITYLEARN_V3_MADRL_REWARD_PROFILES,
# Cierra la estructura de datos o llamada definida arriba.
)

# Configura o actualiza `axis_reward_table`.
axis_reward_table = pd.DataFrame.from_dict(CITYLEARN_V3_AXIS_REWARD_WEIGHTS, orient='index')
# Configura o actualiza `axis_reward_table.index.name`.
axis_reward_table.index.name = 'scenario'
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(axis_reward_table)

# Configura o actualiza `profile_rows`.
profile_rows = []
# Itera sobre una coleccion de elementos.
for algorithm, profile in CITYLEARN_V3_MADRL_REWARD_PROFILES.items():
    # Evalua una condicion antes de continuar el flujo.
    if algorithm == 'MADRL':
        # Ejecuta una instruccion necesaria para esta celda.
        continue
    # Configura o actualiza `multipliers`.
    multipliers = profile['axis_weight_multipliers']
    # Ejecuta una instruccion necesaria para esta celda.
    profile_rows.append({
        # Ejecuta una instruccion necesaria para esta celda.
        'algorithm': algorithm,
        # Ejecuta una instruccion necesaria para esta celda.
        'profile_name': profile['profile_name'],
        # Ejecuta una instruccion necesaria para esta celda.
        'flex_multiplier': multipliers['flex'],
        # Ejecuta una instruccion necesaria para esta celda.
        'carbon_multiplier': multipliers['carbon'],
        # Ejecuta una instruccion necesaria para esta celda.
        'cost_multiplier': multipliers['cost'],
        # Ejecuta una instruccion necesaria para esta celda.
        'team_reward_ratio': profile['team_reward_ratio'],
        # Ejecuta una instruccion necesaria para esta celda.
        'ev_weight': profile['ev_weight'],
        # Ejecuta una instruccion necesaria para esta celda.
        'reward_scale': profile['reward_scale'],
        # Ejecuta una instruccion necesaria para esta celda.
        'peak_weight': profile['peak_weight'],
        # Ejecuta una instruccion necesaria para esta celda.
        'ramp_weight': profile['ramp_weight'],
    # Cierra la estructura de datos o llamada definida arriba.
    })

# Configura o actualiza `reward_profile_table`.
reward_profile_table = pd.DataFrame(profile_rows)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(reward_profile_table)

In [ ]:
# Importa tipos usados por la funcion.
from typing import Dict

# Define los escenarios disponibles antes de la celda global de configuracion.
SCENARIOS = ['E1', 'E2', 'E3']

# Define la funcion auxiliar `effective_axis_weights`.
def effective_axis_weights(scenario: str, algorithm: str) -> Dict[str, float]:
    # Configura o actualiza `base`.
    base = CITYLEARN_V3_AXIS_REWARD_WEIGHTS[scenario]
    # Configura o actualiza `multipliers`.
    multipliers = CITYLEARN_V3_MADRL_REWARD_PROFILES[algorithm]['axis_weight_multipliers']
    # Configura o actualiza `weighted`.
    weighted = {
        # Ejecuta una instruccion necesaria para esta celda.
        key: base[key] * multipliers[key]
        # Itera sobre una coleccion de elementos.
        for key in ['flex', 'carbon', 'cost']
    # Cierra la estructura de datos o llamada definida arriba.
    }
    # Configura o actualiza `total`.
    total = sum(weighted.values())
    # Retorna el resultado calculado por la funcion.
    return {
        # Ejecuta una instruccion necesaria para esta celda.
        key: weighted[key] / total
        # Itera sobre una coleccion de elementos.
        for key in weighted
    # Cierra la estructura de datos o llamada definida arriba.
    }


# Configura o actualiza `effective_rows`.
effective_rows = []
# Itera sobre una coleccion de elementos.
for algorithm in ['HAPPO', 'MASAC', 'MATD3', 'MAAC']:
    # Itera sobre una coleccion de elementos.
    for scenario in SCENARIOS:
        # Configura o actualiza `row`.
        row = {
            # Ejecuta una instruccion necesaria para esta celda.
            'algorithm': algorithm,
            # Ejecuta una instruccion necesaria para esta celda.
            'scenario': scenario,
            # Ejecuta una instruccion necesaria para esta celda.
            **effective_axis_weights(scenario, algorithm),
        # Cierra la estructura de datos o llamada definida arriba.
        }
        # Ejecuta una instruccion necesaria para esta celda.
        effective_rows.append(row)

# Configura o actualiza `effective_reward_table`.
effective_reward_table = pd.DataFrame(effective_rows)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(effective_reward_table)

## CityLearn

CityLearn v3 no reemplaza CityLearn v2. Lo envuelve.

- CityLearn v2 conserva datasets, fisica, evaluacion y API base.
- CityLearn v3 agrega adaptadores Dec-POMDP, reportes multiobjetivo, backends oficiales y estructura de artefactos.
- Los algoritmos MADRL viven en `external/`, no dentro de `citylearn.agents`.


### Environment

El entorno principal es `CityLearnDecPOMDPEnv`, compatible con la idea de `ParallelEnv`: cada agente recibe su observacion y devuelve su accion. La propiedad `state()` da el estado global usado por algoritmos CTDE.


### Control

| Algoritmo | Entrenamiento | Ejecucion |
|---|---|---|
| HAPPO | critico centralizado HARL | actor por edificio |
| MASAC | estado global estilo SMAC | accion discreta mapeada a CityLearn |
| MATD3 | critico con observaciones/acciones conjuntas | actor continuo por edificio |
| MAAC | critico de atencion multiagente | politica local por edificio |


### Backends oficiales, GitHub y papers

| Algoritmo | Paper base | GitHub oficial / backend usado | Rol en este proyecto |
|---|---|---|---|
| HAPPO | Zhong et al. (2024), *Heterogeneous-Agent Reinforcement Learning*, JMLR. https://jmlr.org/papers/v25/23-0488.html | HARL: https://github.com/PKU-MARL/HARL | Backend principal para HAPPO con políticas heterogéneas y critic centralizado. |
| MASAC / mSAC | Pu et al. (2021), *Decomposed Soft Actor-Critic Method for Cooperative Multi-Agent Reinforcement Learning*. https://arxiv.org/abs/2104.06655 | MARL: https://github.com/puyuan1996/MARL | Backend paper-repository para mSAC/MASAC cooperativo. |
| MATD3 | Ackermann et al. (2019), *Reducing Overestimation Bias in Multi-Agent Domains Using Double Centralized Critics*. https://arxiv.org/abs/1910.01465 | Repositorio original: https://github.com/JohannesAck/MATD3implementation; backend PyTorch usado: https://github.com/marlbenchmark/off-policy | El repo original es referencia oficial; para Python 3.9 se usa backend PyTorch source-backed. |
| MAAC | Iqbal y Sha (2019), *Actor-Attention-Critic for Multi-Agent Reinforcement Learning*. https://arxiv.org/abs/1810.02912 | MAAC: https://github.com/shariqiqbal2810/MAAC | Backend original con crítico de atención multiagente. |
| MARLlib | Hu et al. (2023), *MARLlib: A Scalable and Efficient Multi-agent Reinforcement Learning Library*. https://arxiv.org/abs/2210.13708 | MARLlib: https://github.com/Replicable-MARL/MARLlib | Framework MARL adicional para registro del entorno `citylearn_v3`. |

Regla metodológica: ningún algoritmo MADRL se reimplementa dentro de `citylearn.agents`. Cada entrenamiento debe llamar el backend externo correspondiente y registrar en `results.json` el backend, hiperparámetros, checkpoints y artefactos de evaluación.


### Datasets

El proyecto puede usar cualquier `schema.json` CityLearn v2 compatible. El caso de tesis usa `citylearn_challenge_2022_phase_all_plus_evs` con 17 edificios y EVs.


### Other Environments

MARLlib queda registrado mediante un adaptador `citylearn_v3`. Los backends oficiales se conservan como fuentes externas:

- `external/HARL`
- `external/MARL`
- `external/off-policy`
- `external/MAAC`
- `external/MARLlib`
- `external/MATD3implementation` como referencia legacy del paper MATD3


## Other References

Consulta tambien `ESTRATEGIA_3PILARES_MADRL.md`, `CityLearn/CITYLEARN_V3_MADRL.md`, `external/backends.lock.json` y los manifiestos de figuras en `outputs/<experimento>/<madrl>/<escenario>_seed_<seed>/figures/`.


# Hands-On Experiments

Las siguientes celdas estan disenadas para ejecutarse dentro del repositorio completo `MADRLCitytleranflexresdr`. Algunas son de inspeccion rapida y otras lanzan entrenamientos. Por defecto, las celdas largas quedan protegidas con banderas booleanas.


<a name="software-requirements"></a>

# Software Requirements

Primero verificamos la version de Python. El proyecto fue preparado para Python 3.9.


In [ ]:
# Ejecuta un comando de shell/notebook para preparar o verificar el entorno.
!python --version

Si estas en una maquina nueva, instala dependencias desde el entorno preparado del proyecto. En este repositorio ya se usa `.venv39-citylearn-v3`; en Colab tendrias que clonar el repositorio con submodulos y recrear el entorno.


In [ ]:
# Ejemplo local, no ejecutar si el entorno ya esta preparado:
# !python -m pip install -e ../CityLearn pytest matplotlib pandas numpy
# !python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Importamos librerias comunes y configuramos rutas. La funcion `find_project_root` permite ejecutar el notebook desde la raiz del proyecto o desde `CityLearn/examples`.


In [ ]:
# Importa dependencias necesarias para esta seccion.
from __future__ import annotations

# Importa dependencias necesarias para esta seccion.
import json
# Importa dependencias necesarias para esta seccion.
import subprocess
# Importa dependencias necesarias para esta seccion.
import sys
# Importa dependencias necesarias para esta seccion.
from pathlib import Path
# Importa dependencias necesarias para esta seccion.
from typing import Dict, List, Mapping, Optional, Sequence

# Importa dependencias necesarias para esta seccion.
import numpy as np
# Importa dependencias necesarias para esta seccion.
import pandas as pd
# Importa dependencias necesarias para esta seccion.
import matplotlib.pyplot as plt
# Importa dependencias necesarias para esta seccion.
from IPython.display import Image, Markdown, display


# Define la funcion auxiliar `find_project_root`.
def find_project_root(start: Optional[Path] = None) -> Path:
    # Configura o actualiza `start`.
    start = Path.cwd() if start is None else Path(start).resolve()
    # Configura o actualiza `candidates`.
    candidates = [start, *start.parents]

    # Itera sobre una coleccion de elementos.
    for candidate in candidates:
        # Evalua una condicion antes de continuar el flujo.
        if (candidate / 'CityLearn').exists() and (candidate / 'external').exists():
            # Retorna el resultado calculado por la funcion.
            return candidate

    # Itera sobre una coleccion de elementos.
    for candidate in candidates:
        # Evalua una condicion antes de continuar el flujo.
        if (candidate / 'citylearn').exists() and (candidate / 'examples').exists():
            # Retorna el resultado calculado por la funcion.
            return candidate.parent if candidate.name == 'CityLearn' else candidate

    # Retorna el resultado calculado por la funcion.
    return start


# Configura o actualiza `PROJECT_ROOT`.
PROJECT_ROOT = find_project_root()
# Configura o actualiza `CITYLEARN_ROOT`.
CITYLEARN_ROOT = PROJECT_ROOT / 'CityLearn' if (PROJECT_ROOT / 'CityLearn').exists() else PROJECT_ROOT
# Configura o actualiza `EXTERNAL_ROOT`.
EXTERNAL_ROOT = PROJECT_ROOT / 'external'
# Configura o actualiza `SCRIPTS_DIR`.
SCRIPTS_DIR = CITYLEARN_ROOT / 'scripts'

# Itera sobre una coleccion de elementos.
for path in [PROJECT_ROOT, CITYLEARN_ROOT, SCRIPTS_DIR]:
    # Evalua una condicion antes de continuar el flujo.
    if str(path) not in sys.path:
        # Ejecuta una instruccion necesaria para esta celda.
        sys.path.insert(0, str(path))

# Muestra informacion de seguimiento para el usuario.
print('PROJECT_ROOT =', PROJECT_ROOT)
# Muestra informacion de seguimiento para el usuario.
print('CITYLEARN_ROOT =', CITYLEARN_ROOT)
# Muestra informacion de seguimiento para el usuario.
print('EXTERNAL_ROOT =', EXTERNAL_ROOT)

Aqui incluimos ajustes globales para el resto del notebook. Para tutorial se usa un horizonte corto; para tesis se usa `8760` pasos por episodio.


In [ ]:
# Configura o dibuja una visualizacion.
plt.rcParams['figure.figsize'] = (10, 4)
# Configura o dibuja una visualizacion.
plt.rcParams['axes.grid'] = True
# Configura o dibuja una visualizacion.
plt.rcParams['grid.alpha'] = 0.25
# Ejecuta una instruccion necesaria para esta celda.
pd.set_option('display.max_columns', 120)

# Configura o actualiza `RANDOM_SEED`.
RANDOM_SEED = 0
# Configura o actualiza `SCENARIOS`.
SCENARIOS = ['E1', 'E2', 'E3']
# Configura o actualiza `SCENARIO`.
SCENARIO = 'E1'  # escenario corto para celdas interactivas del tutorial
# Configura o actualiza `TUTORIAL_ALGORITHM`.
TUTORIAL_ALGORITHM = 'HAPPO'  # perfil reward v3 usado en celdas interactivas
# Configura o actualiza `OFFICIAL_SCENARIO`.
OFFICIAL_SCENARIO = 'ALL'  # ejecuta E1, E2 y E3 en el launcher oficial
# Configura o actualiza `TUTORIAL_EPISODE_TIME_STEPS`.
TUTORIAL_EPISODE_TIME_STEPS = 24
# Configura o actualiza `OFFICIAL_EPISODE_TIME_STEPS`.
OFFICIAL_EPISODE_TIME_STEPS = 8760
# Configura o actualiza `ALGORITHMS`.
ALGORITHMS = ['happo', 'masac', 'matd3', 'maac']

<a name="data-description"></a>

# Dataset Description

## Loading the Data

El dataset de tesis es una variante CityLearn v2 con 17 edificios y EVs. Se carga desde su `schema.json`, igual que en el tutorial original se carga el dataset base de CityLearn.


In [ ]:
# Importa dependencias necesarias para esta seccion.
from citylearn.data import DataSet
# Importa dependencias necesarias para esta seccion.
from citylearn.dec_pomdp import DEFAULT_17_BUILDING_EV_SCHEMA

# Configura o actualiza `DATASET_NAME`.
DATASET_NAME = 'citylearn_challenge_2022_phase_all_plus_evs'
# Configura o actualiza `SCHEMA_PATH`.
SCHEMA_PATH = Path(DEFAULT_17_BUILDING_EV_SCHEMA)

# Configura o actualiza `schema`.
schema = json.loads(SCHEMA_PATH.read_text(encoding='utf-8'))
# Muestra informacion de seguimiento para el usuario.
print('Dataset:', DATASET_NAME)
# Muestra informacion de seguimiento para el usuario.
print('Schema:', SCHEMA_PATH)
# Muestra informacion de seguimiento para el usuario.
print('Simulation start:', schema.get('simulation_start_time_step'))
# Muestra informacion de seguimiento para el usuario.
print('Simulation end:', schema.get('simulation_end_time_step'))
# Muestra informacion de seguimiento para el usuario.
print('Buildings:', len(schema.get('buildings', {})))

Podemos listar los datasets disponibles en CityLearn. El dataset de tesis puede no aparecer si es una extension local del proyecto, pero su `schema.json` esta dentro de `CityLearn/data/datasets`.


In [ ]:
# Inicia un bloque protegido para capturar errores controlados.
try:
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(pd.Series(sorted(DataSet.get_names()), name='dataset').to_frame())
# Captura una excepcion para manejarla sin detener todo el notebook.
except Exception as exc:
    # Muestra informacion de seguimiento para el usuario.
    print('Could not list DataSet names:', exc)

### Preview a Building Data File

Inspeccionamos el primer edificio incluido. En CityLearn v2 cada edificio apunta a archivos CSV de cargas, clima, precios e intensidad de carbono.


In [ ]:
# Define la funcion auxiliar `included_buildings`.
def included_buildings(schema: Mapping[str, object]) -> List[str]:
    # Configura o actualiza `buildings`.
    buildings = schema.get('buildings', {})
    # Retorna el resultado calculado por la funcion.
    return [name for name, payload in buildings.items() if payload.get('include', True)]


# Define la funcion auxiliar `resolve_dataset_file`.
def resolve_dataset_file(schema_path: Path, schema: Mapping[str, object], filename: str) -> Path:
    # Configura o actualiza `root`.
    root = Path(schema.get('root_directory') or schema_path.parent)
    # Evalua una condicion antes de continuar el flujo.
    if not root.is_absolute():
        # Configura o actualiza `direct`.
        direct = schema_path.parent / root
        # Configura o actualiza `root`.
        root = direct if direct.exists() else schema_path.parent
    # Retorna el resultado calculado por la funcion.
    return root / filename


# Configura o actualiza `building_names`.
building_names = included_buildings(schema)
# Configura o actualiza `building_name`.
building_name = building_names[0]
# Configura o actualiza `building_schema`.
building_schema = schema['buildings'][building_name]
# Muestra informacion de seguimiento para el usuario.
print('Selected building:', building_name)
# Muestra informacion de seguimiento para el usuario.
print('Building keys:', sorted(building_schema.keys()))

# Configura o actualiza `building_file`.
building_file = resolve_dataset_file(SCHEMA_PATH, schema, building_schema['energy_simulation'])
# Configura o actualiza `building_data`.
building_data = pd.read_csv(building_file)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(building_data.head())
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(building_data.describe(include='all').T.head(20))

El archivo del edificio permite revisar cargas no desplazables, condiciones interiores, refrigeracion/calefaccion y otros perfiles. Estas variables son la base fisica que los agentes no deben inventar: el entrenamiento MADRL opera sobre la misma simulacion CityLearn v2.


In [ ]:
# Configura o actualiza `columns`.
columns = [col for col in building_data.columns if any(token in col.lower() for token in ['load', 'cooling', 'heating', 'temperature'])]
# Configura o actualiza `columns`.
columns = columns[:4]
# Configura o dibuja una visualizacion.
fig, axes = plt.subplots(len(columns), 1, figsize=(11, max(3, 2.2 * len(columns))), sharex=True)
# Evalua una condicion antes de continuar el flujo.
if len(columns) == 1:
    # Configura o dibuja una visualizacion.
    axes = [axes]
# Itera sobre una coleccion de elementos.
for ax, column in zip(axes, columns):
    # Configura o dibuja una visualizacion.
    ax.plot(building_data[column].iloc[:168].values, linewidth=1.2)
    # Configura o dibuja una visualizacion.
    ax.set_title(column)
    # Configura o dibuja una visualizacion.
    ax.set_xlabel('hour')
# Ejecuta una instruccion necesaria para esta celda.
fig.tight_layout()
# Configura o dibuja una visualizacion.
plt.show()

### Preview Weather File

El clima afecta cargas termicas y produccion PV. Revisarlo ayuda a entender por que el aprendizaje debe generalizar por episodios y semillas.


In [ ]:
# Configura o actualiza `weather_file`.
weather_file = resolve_dataset_file(SCHEMA_PATH, schema, building_schema['weather'])
# Configura o actualiza `weather_data`.
weather_data = pd.read_csv(weather_file)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(weather_data.head())
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(weather_data.describe(include='all').T.head(20))

# Configura o actualiza `weather_columns`.
weather_columns = [col for col in weather_data.columns if any(token in col.lower() for token in ['temperature', 'humidity', 'solar', 'radiation'])][:4]
# Configura o dibuja una visualizacion.
fig, axes = plt.subplots(len(weather_columns), 1, figsize=(11, max(3, 2.2 * len(weather_columns))), sharex=True)
# Evalua una condicion antes de continuar el flujo.
if len(weather_columns) == 1:
    # Configura o dibuja una visualizacion.
    axes = [axes]
# Itera sobre una coleccion de elementos.
for ax, column in zip(axes, weather_columns):
    # Configura o dibuja una visualizacion.
    ax.plot(weather_data[column].iloc[:168].values, linewidth=1.2)
    # Configura o dibuja una visualizacion.
    ax.set_title(column)
# Ejecuta una instruccion necesaria para esta celda.
fig.tight_layout()
# Configura o dibuja una visualizacion.
plt.show()

### Preview Electricity Price Data

El eje OE3 depende de costos y tarifas dinamicas. Esta serie se usa para evaluar si la politica desplaza consumo hacia horas economicamente favorables.


In [ ]:
# Configura o actualiza `pricing_file`.
pricing_file = resolve_dataset_file(SCHEMA_PATH, schema, building_schema['pricing'])
# Configura o actualiza `pricing_data`.
pricing_data = pd.read_csv(pricing_file)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pricing_data.head())
# Configura o actualiza `pricing_column`.
pricing_column = pricing_data.columns[0]

# Configura o dibuja una visualizacion.
fig, ax = plt.subplots(figsize=(11, 2.8))
# Configura o dibuja una visualizacion.
ax.plot(pricing_data[pricing_column].iloc[:168].values, color='tab:purple', linewidth=1.2)
# Configura o dibuja una visualizacion.
ax.set_title(f'Electricity price: {pricing_column}')
# Configura o dibuja una visualizacion.
ax.set_xlabel('hour')
# Configura o dibuja una visualizacion.
plt.show()

### Preview Carbon Intensity Data

El eje OE2 usa intensidad de carbono para calcular emisiones y comparar control contra baseline. Si el dataset expone esta serie, CityLearn v2 calcula KPIs de carbono.


In [ ]:
# Configura o actualiza `carbon_filename`.
carbon_filename = building_schema.get('carbon_intensity')
# Evalua una condicion antes de continuar el flujo.
if carbon_filename:
    # Configura o actualiza `carbon_file`.
    carbon_file = resolve_dataset_file(SCHEMA_PATH, schema, carbon_filename)
    # Configura o actualiza `carbon_data`.
    carbon_data = pd.read_csv(carbon_file)
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(carbon_data.head())
    # Configura o actualiza `carbon_column`.
    carbon_column = carbon_data.columns[0]
    # Configura o dibuja una visualizacion.
    fig, ax = plt.subplots(figsize=(11, 2.8))
    # Configura o dibuja una visualizacion.
    ax.plot(carbon_data[carbon_column].iloc[:168].values, color='tab:green', linewidth=1.2)
    # Configura o dibuja una visualizacion.
    ax.set_title(f'Carbon intensity: {carbon_column}')
    # Configura o dibuja una visualizacion.
    ax.set_xlabel('hour')
    # Configura o dibuja una visualizacion.
    plt.show()
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('This building does not define a carbon_intensity file.')

## Data Preprocessing

Igual que el tutorial original, podemos modificar una copia del `schema` para seleccionar edificios, periodo de simulacion y observaciones. Para la tesis, el entrenamiento oficial usa los 17 edificios y el horizonte completo.


In [ ]:
# Define la funcion auxiliar `set_schema_buildings`.
def set_schema_buildings(schema: Dict[str, object], count: int) -> Dict[str, object]:
    # Configura o actualiza `output`.
    output = json.loads(json.dumps(schema))
    # Configura o actualiza `names`.
    names = list(output.get('buildings', {}).keys())
    # Configura o actualiza `selected`.
    selected = set(names[:count])
    # Itera sobre una coleccion de elementos.
    for name, payload in output.get('buildings', {}).items():
        # Configura o actualiza `payload['include']`.
        payload['include'] = name in selected
    # Retorna el resultado calculado por la funcion.
    return output


# Define la funcion auxiliar `set_schema_simulation_period`.
def set_schema_simulation_period(schema: Dict[str, object], start: int, end: int) -> Dict[str, object]:
    # Configura o actualiza `output`.
    output = json.loads(json.dumps(schema))
    # Configura o actualiza `output['simulation_start_time_step']`.
    output['simulation_start_time_step'] = int(start)
    # Configura o actualiza `output['simulation_end_time_step']`.
    output['simulation_end_time_step'] = int(end)
    # Retorna el resultado calculado por la funcion.
    return output


# Define la funcion auxiliar `active_observation_names`.
def active_observation_names(schema: Mapping[str, object]) -> List[str]:
    # Configura o actualiza `observations`.
    observations = schema.get('observations', {})
    # Retorna el resultado calculado por la funcion.
    return [name for name, payload in observations.items() if payload.get('active', False)]


# Define la funcion auxiliar `active_action_names`.
def active_action_names(schema: Mapping[str, object]) -> List[str]:
    # Configura o actualiza `actions`.
    actions = schema.get('actions', {})
    # Retorna el resultado calculado por la funcion.
    return [name for name, payload in actions.items() if payload.get('active', False)]

# Muestra informacion de seguimiento para el usuario.
print('Active observations:', active_observation_names(schema)[:25])
# Muestra informacion de seguimiento para el usuario.
print('Active actions:', active_action_names(schema))

### Setting your Random Seed

La semilla controla inicializaciones, escenarios aleatorios y reproducibilidad. En experimentos formales se deben ejecutar varias semillas por algoritmo.


In [ ]:
# Ejecuta una instruccion necesaria para esta celda.
np.random.seed(RANDOM_SEED)
# Muestra informacion de seguimiento para el usuario.
print('RANDOM_SEED =', RANDOM_SEED)

### Setting the Buildings, Time Periods and Observations to use in Simulations from the Schema

Para una ejecucion pedagogica se puede usar un periodo corto. Para resultados oficiales se usa todo el ano y todos los edificios.


In [ ]:
# Configura o actualiza `tutorial_schema`.
tutorial_schema = set_schema_buildings(schema, count=17)
# Configura o actualiza `tutorial_schema`.
tutorial_schema = set_schema_simulation_period(tutorial_schema, start=0, end=TUTORIAL_EPISODE_TIME_STEPS - 1)
# Muestra informacion de seguimiento para el usuario.
print('Tutorial buildings:', len(included_buildings(tutorial_schema)))
# Muestra informacion de seguimiento para el usuario.
print('Tutorial time steps:', tutorial_schema['simulation_start_time_step'], 'to', tutorial_schema['simulation_end_time_step'])
# Muestra informacion de seguimiento para el usuario.
print('Official time steps:', 0, 'to', OFFICIAL_EPISODE_TIME_STEPS - 1)

# Initialize a CityLearn v3 Dec-POMDP Environment

La celda siguiente crea el entorno del proyecto. Internamente se usa `CityLearnEnv` de CityLearn v2, pero la interfaz externa es multiagente.


In [ ]:
# Importa dependencias necesarias para esta seccion.
from citylearn.v3 import (
    # Ejecuta una instruccion necesaria para esta celda.
    describe_environment,
    # Ejecuta una instruccion necesaria para esta celda.
    evaluate_objectives,
    # Ejecuta una instruccion necesaria para esta celda.
    make_citylearn_v3_project_env,
    # Ejecuta una instruccion necesaria para esta celda.
    objective_manifest,
# Cierra la estructura de datos o llamada definida arriba.
)

# Configura o actualiza `env`.
env = make_citylearn_v3_project_env(
    # Configura o actualiza `scenario`.
    scenario=SCENARIO,
    # Configura o actualiza `seed`.
    seed=RANDOM_SEED,
    # Configura o actualiza `episode_time_steps`.
    episode_time_steps=TUTORIAL_EPISODE_TIME_STEPS,
    # Configura o actualiza `madrl_algorithm`.
    madrl_algorithm=TUTORIAL_ALGORITHM,
# Cierra la estructura de datos o llamada definida arriba.
)

# Configura o actualiza `description`.
description = describe_environment(env)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.Series(description).to_frame('value'))
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.Series(description['reward_metadata']).to_frame('reward_metadata'))

El objeto `env` conserva el simulador CityLearn v2 en `env.env`, y expone propiedades multiagente como `possible_agents`, `observation_space(agent)`, `action_space(agent)` y `state()`.


In [ ]:
# Muestra informacion de seguimiento para el usuario.
print('Number of agents:', env.num_agents)
# Muestra informacion de seguimiento para el usuario.
print('First agents:', env.possible_agents[:5])

# Configura o actualiza `space_rows`.
space_rows = []
# Itera sobre una coleccion de elementos.
for agent in env.possible_agents:
    # Ejecuta una instruccion necesaria para esta celda.
    space_rows.append({
        # Ejecuta una instruccion necesaria para esta celda.
        'agent': agent,
        # Ejecuta una instruccion necesaria para esta celda.
        'observation_dim': int(env.observation_space(agent).shape[0]),
        # Ejecuta una instruccion necesaria para esta celda.
        'action_dim': int(env.action_space(agent).shape[0]),
    # Cierra la estructura de datos o llamada definida arriba.
    })

# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.DataFrame(space_rows))
# Muestra informacion de seguimiento para el usuario.
print('CTDE state dimension:', env.state_space.shape)

Ejecutamos unos pasos con acciones cero para confirmar la forma de observaciones, recompensas, terminaciones y estado global. Este no es entrenamiento: es una prueba funcional del Dec-POMDP.


In [ ]:
# Configura o actualiza `infos`.
observations, infos = env.reset(seed=RANDOM_SEED)
# Muestra informacion de seguimiento para el usuario.
print('Observation keys:', list(observations)[:5])
# Muestra informacion de seguimiento para el usuario.
print('Initial CTDE state shape:', env.state().shape)

# Itera sobre una coleccion de elementos.
for step in range(3):
    # Configura o actualiza `actions`.
    actions = {
        # Configura o actualiza `dtype`.
        agent: np.zeros(env.action_space(agent).shape, dtype=np.float32)
        # Itera sobre una coleccion de elementos.
        for agent in env.agents
    # Cierra la estructura de datos o llamada definida arriba.
    }
    # Configura o actualiza `infos`.
    observations, rewards, terminations, truncations, infos = env.step(actions)
    # Muestra informacion de seguimiento para el usuario.
    print(f'step={step}', 'reward_mean=', np.mean(list(rewards.values())), 'active_agents=', len(env.agents))

# Key Performance Indicators for Evaluation

CityLearn v2 produce KPIs de evaluacion. CityLearn v3 organiza esos KPIs en tres ejes de tesis. Esta separacion evita confundir metricas de simulacion con objetivos cientificos.


In [ ]:
# Configura o actualiza `objective_info`.
objective_info = objective_manifest()
# Itera sobre una coleccion de elementos.
for axis, payload in objective_info['axes'].items():
    # Muestra informacion de seguimiento para el usuario.
    print(axis, '-', payload['name'])
    # Muestra informacion de seguimiento para el usuario.
    print(' ', payload['statement'])
    # Muestra informacion de seguimiento para el usuario.
    print(' ', 'kpi_count =', len(payload['kpis']))

# Configura o actualiza `axis_kpi_rows`.
axis_kpi_rows = []
# Itera sobre una coleccion de elementos.
for axis, payload in objective_info['axes'].items():
    # Itera sobre una coleccion de elementos.
    for kpi in payload['kpis']:
        # Configura o actualiza `trace`.
        trace = objective_info['axis_kpis'].get(kpi, {})
        # Ejecuta una instruccion necesaria para esta celda.
        axis_kpi_rows.append({
            # Ejecuta una instruccion necesaria para esta celda.
            'axis': axis,
            # Ejecuta una instruccion necesaria para esta celda.
            'axis_name': payload['name'],
            # Ejecuta una instruccion necesaria para esta celda.
            'scenario': payload['scenario'],
            # Ejecuta una instruccion necesaria para esta celda.
            'kpi': kpi,
            # Ejecuta una instruccion necesaria para esta celda.
            'source': trace.get('source'),
            # Ejecuta una instruccion necesaria para esta celda.
            'lower_is_better': trace.get('lower_is_better'),
            # Ejecuta una instruccion necesaria para esta celda.
            'citylearn_v2_names': ', '.join(trace.get('citylearn_v2_names', [])),
            # Ejecuta una instruccion necesaria para esta celda.
            'note': trace.get('note', ''),
        # Cierra la estructura de datos o llamada definida arriba.
        })

# Configura o actualiza `axis_kpi_manifest`.
axis_kpi_manifest = pd.DataFrame(axis_kpi_rows)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(axis_kpi_manifest)

In [ ]:
# Configura o actualiza `report`.
report = evaluate_objectives(env)
# Muestra informacion de seguimiento para el usuario.
print('KPI frame rows:', report.get('kpi_frame_rows'))
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.DataFrame([
    # Ejecuta una instruccion necesaria para esta celda.
    {'kpi': name, 'value': value}
    # Itera sobre una coleccion de elementos.
    for name, value in report.get('axis_kpis', {}).items()
# Cierra la estructura de datos o llamada definida arriba.
]).head(30))

# Convenience Functions to Display Simulation Results

El tutorial original define funciones de visualizacion para KPIs, perfiles de carga y baterias. Aqui definimos funciones equivalentes para los artefactos MADRL: resultados, series temporales, trazas por agente y figuras generadas.


In [ ]:
# Define la funcion auxiliar `run_dir`.
def run_dir(output_root: Path, algorithm: str, scenario: str = SCENARIO, seed: int = RANDOM_SEED) -> Path:
    # Retorna el resultado calculado por la funcion.
    return output_root / algorithm.lower() / f'{scenario}_seed_{seed}'


# Define la funcion auxiliar `load_json`.
def load_json(path: Path) -> Dict[str, object]:
    # Retorna el resultado calculado por la funcion.
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {}


# Define la funcion auxiliar `load_run_results`.
def load_run_results(path: Path) -> Dict[str, object]:
    # Retorna el resultado calculado por la funcion.
    return load_json(path / 'data' / 'results.json') or load_json(path / 'results.json')


# Define la funcion auxiliar `load_run_timeseries`.
def load_run_timeseries(path: Path) -> pd.DataFrame:
    # Configura o actualiza `csv_path`.
    csv_path = path / 'data' / 'timeseries.csv'
    # Evalua una condicion antes de continuar el flujo.
    if not csv_path.is_file():
        # Configura o actualiza `csv_path`.
        csv_path = path / 'timeseries.csv'
    # Retorna el resultado calculado por la funcion.
    return pd.read_csv(csv_path) if csv_path.is_file() else pd.DataFrame()


# Define la funcion auxiliar `load_run_trace`.
def load_run_trace(path: Path) -> pd.DataFrame:
    # Configura o actualiza `csv_path`.
    csv_path = path / 'data' / 'trace.csv'
    # Evalua una condicion antes de continuar el flujo.
    if not csv_path.is_file():
        # Configura o actualiza `csv_path`.
        csv_path = path / 'trace.csv'
    # Retorna el resultado calculado por la funcion.
    return pd.read_csv(csv_path) if csv_path.is_file() else pd.DataFrame()


# Define la funcion auxiliar `load_objective_kpis`.
def load_objective_kpis(path: Path) -> pd.DataFrame:
    # Configura o actualiza `table_path`.
    table_path = path / 'figures' / 'tables' / 'objective_kpis.csv'
    # Retorna el resultado calculado por la funcion.
    return pd.read_csv(table_path) if table_path.is_file() else pd.DataFrame()

In [ ]:
# Define la funcion auxiliar `plot_axis_kpis`.
def plot_axis_kpis(results: Mapping[str, object], axis: str) -> plt.Figure:
    # Configura o actualiza `report`.
    report = results.get('citylearn_v3_report', results)
    # Configura o actualiza `axis_payload`.
    axis_payload = report.get('objective_axis_kpis', {}).get(axis, {})
    # Configura o actualiza `rows`.
    rows = []
    # Itera sobre una coleccion de elementos.
    for name, payload in axis_payload.get('kpis', {}).items():
        # Configura o actualiza `value`.
        value = payload.get('value') if isinstance(payload, Mapping) else None
        # Evalua una condicion antes de continuar el flujo.
        if value is not None:
            # Ejecuta una instruccion necesaria para esta celda.
            rows.append({'kpi': name, 'value': value})
    # Configura o actualiza `data`.
    data = pd.DataFrame(rows)
    # Configura o dibuja una visualizacion.
    fig, ax = plt.subplots(figsize=(10, max(3, 0.3 * len(data))))
    # Evalua una condicion antes de continuar el flujo.
    if not data.empty:
        # Configura o dibuja una visualizacion.
        ax.barh(data['kpi'], data['value'])
        # Configura o dibuja una visualizacion.
        ax.set_title(f'{axis} KPI profile')
        # Configura o dibuja una visualizacion.
        ax.set_xlabel('value')
    # Retorna el resultado calculado por la funcion.
    return fig


# Define la funcion auxiliar `plot_district_timeseries`.
def plot_district_timeseries(timeseries: pd.DataFrame) -> plt.Figure:
    # Configura o actualiza `columns`.
    columns = [
        # Ejecuta una instruccion necesaria para esta celda.
        'district_net_electricity_consumption',
        # Ejecuta una instruccion necesaria para esta celda.
        'district_net_electricity_consumption_without_storage',
        # Ejecuta una instruccion necesaria para esta celda.
        'district_net_electricity_consumption_cost',
        # Ejecuta una instruccion necesaria para esta celda.
        'district_net_electricity_consumption_emission',
        # Ejecuta una instruccion necesaria para esta celda.
        'electricity_price_mean',
        # Ejecuta una instruccion necesaria para esta celda.
        'carbon_intensity_mean',
    # Cierra la estructura de datos o llamada definida arriba.
    ]
    # Configura o actualiza `columns`.
    columns = [column for column in columns if column in timeseries.columns]
    # Configura o dibuja una visualizacion.
    fig, axes = plt.subplots(len(columns), 1, figsize=(11, max(3, 2 * len(columns))), sharex=True)
    # Evalua una condicion antes de continuar el flujo.
    if len(columns) == 1:
        # Configura o dibuja una visualizacion.
        axes = [axes]
    # Itera sobre una coleccion de elementos.
    for ax, column in zip(axes, columns):
        # Configura o dibuja una visualizacion.
        ax.plot(timeseries['global_step'], timeseries[column], linewidth=1.1)
        # Configura o dibuja una visualizacion.
        ax.set_title(column)
    # Configura o dibuja una visualizacion.
    axes[-1].set_xlabel('global_step')
    # Ejecuta una instruccion necesaria para esta celda.
    fig.tight_layout()
    # Retorna el resultado calculado por la funcion.
    return fig


# Define la funcion auxiliar `display_generated_figures`.
def display_generated_figures(path: Path, names: Optional[Sequence[str]] = None) -> None:
    # Configura o actualiza `manifest_path`.
    manifest_path = path / 'figures' / 'figures_manifest.json'
    # Configura o actualiza `manifest`.
    manifest = load_json(manifest_path)
    # Configura o actualiza `figures`.
    figures = manifest.get('figures', [])
    # Evalua una condicion antes de continuar el flujo.
    if names is not None:
        # Configura o actualiza `figures`.
        figures = [item for item in figures if item.get('name') in set(names)]
    # Itera sobre una coleccion de elementos.
    for item in figures:
        # Renderiza una tabla, figura o Markdown dentro del notebook.
        display(Markdown(f"### {item.get('name')}"))
        # Renderiza una tabla, figura o Markdown dentro del notebook.
        display(Image(filename=item['path']))

# Build your Baseline Validation

Para comparacion formal, CityLearn v2 calcula KPIs contra una linea base. En CityLearn v3 no se redefine esa linea base; se reutilizan los valores `baseline`, `control`, `delta` y ratios que provienen de `evaluate_v2`.


In [ ]:
# Configura o actualiza `VALIDATE_OBJECTIVES`.
VALIDATE_OBJECTIVES = False
# Configura o actualiza `VALIDATION_OUTPUT`.
VALIDATION_OUTPUT = PROJECT_ROOT / 'outputs' / 'citylearn_v3_objective_validation_notebook'

# Evalua una condicion antes de continuar el flujo.
if VALIDATE_OBJECTIVES:
    # Construye o ejecuta comandos externos del flujo de trabajo.
    command = [
        # Ejecuta una instruccion necesaria para esta celda.
        sys.executable,
        # Ejecuta una instruccion necesaria para esta celda.
        '-B',
        # Ejecuta una instruccion necesaria para esta celda.
        str(SCRIPTS_DIR / 'validate_citylearn_v3_objectives.py'),
        # Declara un argumento o componente del comando ejecutable.
        '--scenario', SCENARIO,
        # Declara un argumento o componente del comando ejecutable.
        '--seed', str(RANDOM_SEED),
        # Declara un argumento o componente del comando ejecutable.
        '--episode-time-steps', str(TUTORIAL_EPISODE_TIME_STEPS),
        # Declara un argumento o componente del comando ejecutable.
        '--include-citylearn-v2-test-agents',
        # Declara un argumento o componente del comando ejecutable.
        '--output-dir', str(VALIDATION_OUTPUT),
    # Cierra la estructura de datos o llamada definida arriba.
    ]
    # Muestra informacion de seguimiento para el usuario.
    print('Running:', ' '.join(map(str, command)))
    # Construye o ejecuta comandos externos del flujo de trabajo.
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('Set VALIDATE_OBJECTIVES=True to run the validation script.')

# An Introduction to MADRL Algorithms as Adaptive Controllers

En lugar de construir un RBC interactivo o un agente Q-learning tabular, este proyecto usa cuatro algoritmos MADRL profundos respaldados por repositorios oficiales. Antes de entrenar, revisamos el manifiesto de backends.


In [ ]:
# Importa dependencias necesarias para esta seccion.
from citylearn.v3.backends import citylearn_v3_backend_manifest

# Configura o actualiza `backend_manifest`.
backend_manifest = citylearn_v3_backend_manifest()
# Muestra informacion de seguimiento para el usuario.
print('Version layer:', backend_manifest['version_layer'])
# Muestra informacion de seguimiento para el usuario.
print('Simulator:', backend_manifest['simulator'])
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.DataFrame.from_dict(backend_manifest['backends'], orient='index'))
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.Series(backend_manifest['dec_pomdp']).to_frame('value'))

## Dec-POMDP and CTDE Contract

Cada algoritmo debe cumplir el mismo contrato experimental aunque sus redes internas sean distintas. La comparacion entre algoritmos solo es defendible si el entorno, dataset, horizonte, semillas, KPIs, recompensas y estructura Dec-POMDP permanecen constantes.

En la implementacion vigente de CityLearn v3 MADRL, la cooperacion se fuerza con `reward_aggregation = team_mean`: los 17 edificios reciben una misma recompensa de equipo por paso. La coordinacion y transferencia de informacion ocurren durante el entrenamiento centralizado mediante el estado global CTDE construido como concatenacion de observaciones locales, `share_observation_space`, `get_state()`, criticos centralizados y critic de atencion segun el backend.

La ejecucion sigue siendo descentralizada: cada edificio ejecuta su actor/politica local con su observacion local. Por tanto, este proyecto implementa comunicacion y transferencia de informacion en entrenamiento CTDE, no mensajeria peer-to-peer explicita entre edificios durante la ejecucion.


In [ ]:
# Configura o actualiza `ctde_contract`.
ctde_contract = pd.DataFrame([
    # Define el contrato CTDE de HAPPO dentro de CityLearn v3 MADRL.
    {
        # Identifica el algoritmo evaluado.
        'algorithm': 'HAPPO',
        # Describe el mecanismo de entrenamiento centralizado.
        'centralized_training': 'HARL centralized critic + share_observation_space',
        # Describe el mecanismo de ejecucion descentralizada.
        'decentralized_execution': 'local continuous actor per building',
        # Describe el tipo de accion usada por CityLearn.
        'action_type': 'continuous',
        # Declara la senal cooperativa comun.
        'cooperative_signal': 'team_mean shared reward for all 17 buildings',
        # Describe como se transfiere informacion entre edificios durante entrenamiento.
        'information_transfer': 'global CTDE state repeated as shared observation for centralized critic',
    # Cierra el registro de HAPPO.
    },
    # Define el contrato CTDE de MASAC dentro de CityLearn v3 MADRL.
    {
        # Identifica el algoritmo evaluado.
        'algorithm': 'MASAC',
        # Describe el mecanismo de entrenamiento centralizado.
        'centralized_training': 'SMAC-style global state through get_state()',
        # Describe el mecanismo de ejecucion descentralizada.
        'decentralized_execution': 'local discrete policy mapped to CityLearn action',
        # Describe el tipo de accion usada por CityLearn.
        'action_type': 'discretized continuous action table',
        # Declara la senal cooperativa comun.
        'cooperative_signal': 'team_mean shared reward for all 17 buildings',
        # Describe como se transfiere informacion entre edificios durante entrenamiento.
        'information_transfer': 'state_shape exposes concatenated district observations to the critic',
    # Cierra el registro de MASAC.
    },
    # Define el contrato CTDE de MATD3 dentro de CityLearn v3 MADRL.
    {
        # Identifica el algoritmo evaluado.
        'algorithm': 'MATD3',
        # Describe el mecanismo de entrenamiento centralizado.
        'centralized_training': 'centralized critic with joint observations and joint actions',
        # Describe el mecanismo de ejecucion descentralizada.
        'decentralized_execution': 'local continuous actor per building',
        # Describe el tipo de accion usada por CityLearn.
        'action_type': 'continuous',
        # Declara la senal cooperativa comun.
        'cooperative_signal': 'team_mean shared reward for all 17 buildings',
        # Describe como se transfiere informacion entre edificios durante entrenamiento.
        'information_transfer': 'padded_joint_observation and centralized action context for critic update',
    # Cierra el registro de MATD3.
    },
    # Define el contrato CTDE de MAAC dentro de CityLearn v3 MADRL.
    {
        # Identifica el algoritmo evaluado.
        'algorithm': 'MAAC',
        # Describe el mecanismo de entrenamiento centralizado.
        'centralized_training': 'multi-agent attention critic over all buildings',
        # Describe el mecanismo de ejecucion descentralizada.
        'decentralized_execution': 'local discrete policy mapped to CityLearn action',
        # Describe el tipo de accion usada por CityLearn.
        'action_type': 'discretized continuous action table',
        # Declara la senal cooperativa comun.
        'cooperative_signal': 'team_mean shared reward for all 17 buildings',
        # Describe como se transfiere informacion entre edificios durante entrenamiento.
        'information_transfer': 'attention critic attends to other building-agent embeddings during training',
    # Cierra el registro de MAAC.
    },
# Cierra la tabla de contrato CTDE.
])
# Renderiza la tabla de contrato Dec-POMDP/CTDE.
display(ctde_contract)


## Validate Cooperative and Coordinated MADRL Contract

Esta celda verifica el requisito de que los cuatro MADRL sean cooperativos, coordinados y con transferencia de informacion entre edificios durante entrenamiento CTDE. La prueba crea entornos cortos de dos pasos para `HAPPO`, `MASAC`, `MATD3` y `MAAC` en `E1`, `E2` y `E3`, y comprueba:

- 17 agentes/edificios activos.
- `reward_aggregation = team_mean`.
- Estado global CTDE con dimension igual a la suma de observaciones locales.
- Recompensa compartida identica para todos los edificios en cada paso.
- `team_reward` e `individual_reward` trazables en `infos`.
- Pesos propios MADRL, sin usar pesos base MARL.

La validacion local mas reciente queda guardada en `outputs/citylearn_v3_madrl_official_full_cuda_v2/cooperative_ctde_validation.json`.


In [ ]:
# Configura o actualiza `COOPERATIVE_CTDE_VALIDATION`.
COOPERATIVE_CTDE_VALIDATION = PROJECT_ROOT / 'outputs' / 'citylearn_v3_madrl_official_full_cuda_v2' / 'cooperative_ctde_validation.json'
# Configura o actualiza `RUN_COOPERATIVE_CTDE_VALIDATION`.
RUN_COOPERATIVE_CTDE_VALIDATION = False


# Define la funcion auxiliar `run_cooperative_ctde_validation`.
def run_cooperative_ctde_validation(output_path: Path) -> Dict[str, object]:
    # Importa dependencias necesarias para esta validacion.
    from citylearn.v3 import describe_environment, make_citylearn_v3_project_env

    # Configura o actualiza `rows`.
    rows = []
    # Itera sobre los algoritmos MADRL oficiales del proyecto.
    for algorithm in ['HAPPO', 'MASAC', 'MATD3', 'MAAC']:
        # Itera sobre los tres ejes/escenarios del proyecto.
        for scenario in SCENARIOS:
            # Construye un entorno corto para validar el contrato sin entrenar.
            check_env = make_citylearn_v3_project_env(
                # Configura el escenario evaluado.
                scenario=scenario,
                # Configura la semilla reproducible.
                seed=RANDOM_SEED,
                # Usa dos pasos para validar formas y recompensas sin costo computacional alto.
                episode_time_steps=2,
                # Activa el perfil de recompensa propio del algoritmo.
                madrl_algorithm=algorithm,
            # Cierra la construccion del entorno de validacion.
            )
            # Describe el entorno Dec-POMDP/CTDE creado.
            description = describe_environment(check_env)
            # Verifica que la recompensa sea cooperativa compartida.
            assert description['reward_aggregation'] == 'team_mean', (algorithm, scenario, description['reward_aggregation'])
            # Verifica que existan los 17 edificios/agentes del dataset oficial.
            assert description['num_agents'] == 17, (algorithm, scenario, description['num_agents'])
            # Verifica que el estado CTDE concatene todas las observaciones locales.
            assert description['state_dim'] == sum(description['observation_dims'].values()), (algorithm, scenario)
            # Verifica que no se usen pesos base MARL como recompensa principal.
            assert description['reward_metadata']['not_using_marl_base_weights'] is True, (algorithm, scenario)

            # Reinicia el entorno para ejecutar una transicion controlada.
            observations, infos = check_env.reset(seed=RANDOM_SEED)
            # Construye acciones cero para todos los edificios.
            actions = {
                # Crea un vector de accion nulo con la forma esperada por el agente.
                agent: np.zeros(check_env.action_space(agent).shape, dtype=np.float32)
                # Itera sobre los agentes activos del entorno.
                for agent in check_env.agents
            # Cierra la construccion de acciones cero.
            }
            # Ejecuta un paso para observar recompensa cooperativa e informacion por agente.
            observations, rewards, terminations, truncations, infos = check_env.step(actions)
            # Redondea recompensas para comprobar igualdad numerica entre edificios.
            rounded_rewards = {round(float(value), 8) for value in rewards.values()}
            # Verifica que todos los edificios reciban la misma recompensa de equipo.
            assert len(rounded_rewards) == 1, (algorithm, scenario, rewards)
            # Verifica trazabilidad de recompensa de equipo por agente.
            assert all('team_reward' in info for info in infos.values()), (algorithm, scenario)
            # Verifica trazabilidad de recompensa individual antes de agregacion cooperativa.
            assert all('individual_reward' in info for info in infos.values()), (algorithm, scenario)

            # Agrega el resultado de validacion a la tabla.
            rows.append({
                # Registra el algoritmo validado.
                'algorithm': algorithm,
                # Registra el eje/escenario validado.
                'scenario': scenario,
                # Registra el numero de edificios/agentes.
                'agents': description['num_agents'],
                # Registra el tipo de agregacion cooperativa.
                'reward_aggregation': description['reward_aggregation'],
                # Registra la dimension del estado global CTDE.
                'state_dim': description['state_dim'],
                # Registra la suma de dimensiones locales de observacion.
                'obs_dim_sum': sum(description['observation_dims'].values()),
                # Registra la recompensa compartida observada.
                'shared_reward_value': next(iter(rounded_rewards)),
                # Registra el perfil reward propio del algoritmo.
                'reward_profile': description['reward_metadata'].get('profile'),
                # Registra que no se usan pesos base MARL.
                'not_using_marl_base_weights': description['reward_metadata'].get('not_using_marl_base_weights'),
            # Cierra el registro de validacion.
            })
            # Cierra el entorno corto de validacion.
            check_env.close()

    # Configura o actualiza `payload`.
    payload = {'status': 'passed', 'rows': rows}
    # Crea el directorio de salida si no existe.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # Guarda la validacion como JSON trazable.
    output_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    # Retorna el resultado calculado por la funcion.
    return payload


# Evalua si se debe regenerar la validacion cooperativa CTDE.
if RUN_COOPERATIVE_CTDE_VALIDATION:
    # Ejecuta la validacion corta para los 4 MADRL y 3 ejes.
    cooperative_validation = run_cooperative_ctde_validation(COOPERATIVE_CTDE_VALIDATION)
# Evalua si existe una validacion generada previamente.
elif COOPERATIVE_CTDE_VALIDATION.is_file():
    # Carga la validacion existente desde la salida oficial.
    cooperative_validation = load_json(COOPERATIVE_CTDE_VALIDATION)
# Ejecuta la rama alternativa cuando no existe validacion local.
else:
    # Crea un resultado vacio informativo.
    cooperative_validation = {'status': 'missing', 'rows': []}

# Muestra el estado de la validacion cooperativa CTDE.
print('cooperative_ctde_validation_status:', cooperative_validation.get('status'))
# Convierte las filas de validacion a tabla.
cooperative_validation_table = pd.DataFrame(cooperative_validation.get('rows', []))
# Evalua si hay filas para mostrar.
if not cooperative_validation_table.empty:
    # Renderiza una tabla resumida de cooperacion y CTDE.
    display(cooperative_validation_table[[
        # Muestra el algoritmo validado.
        'algorithm',
        # Muestra el escenario validado.
        'scenario',
        # Muestra el numero de agentes.
        'agents',
        # Muestra la agregacion de recompensa.
        'reward_aggregation',
        # Muestra la dimension global CTDE.
        'state_dim',
        # Muestra la suma de dimensiones locales.
        'obs_dim_sum',
        # Muestra el valor de recompensa compartida observado.
        'shared_reward_value',
        # Muestra la bandera de pesos propios MADRL.
        'not_using_marl_base_weights',
    # Cierra la lista de columnas.
    ]])
# Ejecuta la rama alternativa cuando no hay validacion local.
else:
    # Muestra la ruta esperada para la validacion.
    print('No validation file found at', COOPERATIVE_CTDE_VALIDATION)


## Validate Reward Profiles

Esta validacion confirma que `E1/E2/E3` usan perfiles de recompensa MADRL propios del proyecto CityLearn v3, que los pesos efectivos suman 1 por eje y que no se esta usando la recompensa base MARL como criterio principal.


In [ ]:
# Configura o actualiza `REWARD_PROFILE_VALIDATION`.
REWARD_PROFILE_VALIDATION = PROJECT_ROOT / 'outputs' / 'citylearn_v3_reward_profile_validation.json'
# Configura o actualiza `VALIDATE_REWARD_PROFILES`.
VALIDATE_REWARD_PROFILES = False

# Evalua una condicion antes de continuar el flujo.
if VALIDATE_REWARD_PROFILES:
    # Construye o ejecuta comandos externos del flujo de trabajo.
    command = [
        # Ejecuta una instruccion necesaria para esta celda.
        sys.executable,
        # Ejecuta una instruccion necesaria para esta celda.
        '-B',
        # Ejecuta una instruccion necesaria para esta celda.
        str(SCRIPTS_DIR / 'validate_citylearn_v3_reward_profiles.py'),
        # Declara un argumento o componente del comando ejecutable.
        '--output',
        # Ejecuta una instruccion necesaria para esta celda.
        str(REWARD_PROFILE_VALIDATION),
    # Cierra la estructura de datos o llamada definida arriba.
    ]
    # Muestra informacion de seguimiento para el usuario.
    print('Running:', ' '.join(map(str, command)))
    # Construye o ejecuta comandos externos del flujo de trabajo.
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

# Evalua una condicion antes de continuar el flujo.
if REWARD_PROFILE_VALIDATION.is_file():
    # Configura o actualiza `reward_validation`.
    reward_validation = load_json(REWARD_PROFILE_VALIDATION)
    # Muestra informacion de seguimiento para el usuario.
    print('status:', reward_validation['status'])
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(pd.DataFrame(reward_validation['rows']))
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('No reward profile validation JSON found yet.')

# Optimize MADRL Controllers

Los entrenamientos se ejecutan con scripts separados por algoritmo. En el flujo oficial actual se usa `-Scenario ALL`, que ejecuta secuencialmente los tres ejes `E1`, `E2` y `E3` para cada MADRL. Todos escriben la misma estructura de artefactos:

```text
outputs/<experimento>/<madrl>/<escenario>_seed_<seed>/
  data/
  checkpoints/
  figures/
  live_progress.json
  results.json
  training_summary.json
  timeseries.csv
  trace.csv
```

Por seguridad, las celdas de entrenamiento no se ejecutan automaticamente.


In [ ]:
# Define la funcion auxiliar `train_command`.
def train_command(
    # Ejecuta una instruccion necesaria para esta celda.
    algorithm: str,
    # Ejecuta una instruccion necesaria para esta celda.
    output_root: Path,
    # Ejecuta una instruccion necesaria para esta celda.
    episode_time_steps: int,
    # Ejecuta una instruccion necesaria para esta celda.
    episodes: int,
    # Configura o actualiza `bool`.
    cuda: bool = True,
    # Configura o actualiza `str`.
    scenario: str = SCENARIO,
# Cierra la estructura de datos o llamada definida arriba.
) -> List[str]:
    # Configura o actualiza `script`.
    script = SCRIPTS_DIR / f'train_citylearn_v3_{algorithm}.py'
    # Configura o actualiza `num_env_steps`.
    num_env_steps = episode_time_steps * episodes
    # Construye o ejecuta comandos externos del flujo de trabajo.
    command = [
        # Ejecuta una instruccion necesaria para esta celda.
        sys.executable,
        # Ejecuta una instruccion necesaria para esta celda.
        '-B',
        # Ejecuta una instruccion necesaria para esta celda.
        str(script),
        # Declara un argumento o componente del comando ejecutable.
        '--scenario', scenario,
        # Declara un argumento o componente del comando ejecutable.
        '--seed', str(RANDOM_SEED),
        # Declara un argumento o componente del comando ejecutable.
        '--episode-time-steps', str(episode_time_steps),
        # Declara un argumento o componente del comando ejecutable.
        '--episodes', str(episodes),
        # Declara un argumento o componente del comando ejecutable.
        '--output-dir', str(output_root / algorithm),
    # Cierra la estructura de datos o llamada definida arriba.
    ]
    # Evalua una condicion antes de continuar el flujo.
    if algorithm == 'happo':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--num-env-steps', str(num_env_steps),
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--torch-threads', '12',
            # Declara un argumento o componente del comando ejecutable.
            '--n-rollout-threads', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--log-interval', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--eval-interval', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'masac':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--action-bins', '3',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-size', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--critic-batch-size', '2',
            # Declara un argumento o componente del comando ejecutable.
            '--critic-train-steps', '2',
            # Declara un argumento o componente del comando ejecutable.
            '--actor-sample-times', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--rnn-hidden-dim', '128',
            # Declara un argumento o componente del comando ejecutable.
            '--qmix-hidden-dim', '64',
            # Declara un argumento o componente del comando ejecutable.
            '--hyper-hidden-dim', '128',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'matd3':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--num-env-steps', str(num_env_steps),
            # Declara un argumento o componente del comando ejecutable.
            '--batch-size', '512',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-size', '50000',
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--train-interval', '100',
            # Declara un argumento o componente del comando ejecutable.
            '--num-random-episodes', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'maac':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--action-bins', '3',
            # Declara un argumento o componente del comando ejecutable.
            '--batch-size', '512',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-length', '200000',
            # Declara un argumento o componente del comando ejecutable.
            '--steps-per-update', '250',
            # Declara un argumento o componente del comando ejecutable.
            '--num-updates', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--attend-heads', '4',
            # Declara un argumento o componente del comando ejecutable.
            '--pi-lr', '0.0003',
            # Declara un argumento o componente del comando ejecutable.
            '--q-lr', '0.001',
            # Declara un argumento o componente del comando ejecutable.
            '--tau', '0.005',
            # Declara un argumento o componente del comando ejecutable.
            '--gamma', '0.99',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion antes de continuar el flujo.
    if cuda:
        # Ejecuta una instruccion necesaria para esta celda.
        command.append('--cuda')
    # Retorna el resultado calculado por la funcion.
    return command


# Configura o actualiza `SMOKE_OUTPUT`.
SMOKE_OUTPUT = PROJECT_ROOT / 'outputs' / 'citylearn_v3_notebook_smoke'
# Itera sobre una coleccion de elementos.
for algorithm in ALGORITHMS:
    # Muestra informacion de seguimiento para el usuario.
    print(algorithm.upper())
    # Muestra informacion de seguimiento para el usuario.
    print(' '.join(map(str, train_command(algorithm, SMOKE_OUTPUT, episode_time_steps=4, episodes=1))))

## Train

Esta celda ejecuta entrenamientos minimos. Usala para validar que los cuatro backends arrancan, no para obtener resultados cientificos.


In [ ]:
# Configura o actualiza `RUN_SMOKE_TRAINING`.
RUN_SMOKE_TRAINING = False

# Evalua una condicion antes de continuar el flujo.
if RUN_SMOKE_TRAINING:
    # Itera sobre una coleccion de elementos.
    for algorithm in ALGORITHMS:
        # Construye o ejecuta comandos externos del flujo de trabajo.
        command = train_command(algorithm, SMOKE_OUTPUT, episode_time_steps=4, episodes=1, cuda=True)
        # Muestra informacion de seguimiento para el usuario.
        print('Running', algorithm.upper())
        # Construye o ejecuta comandos externos del flujo de trabajo.
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('Set RUN_SMOKE_TRAINING=True to train a tiny run for all algorithms.')

## Official Full Training

El entrenamiento oficial vigente usa el dataset `citylearn_challenge_2022_phase_all_plus_evs`, 17 edificios + EV, `-Scenario ALL`, 8760 pasos por episodio, 5 episodios y CUDA con PyTorch `2.8.0+cu126`. El lanzador PowerShell ejecuta de forma secuencial la matriz `E1/E2/E3 x HAPPO/MASAC/MATD3/MAAC`; esto reduce conflictos de memoria GPU y deja artefactos separados por algoritmo y eje.

La salida oficial relanzada desde cero se guarda en:

```text
outputs/citylearn_v3_madrl_official_full_cuda_v2/
  official_full_status.json
  cooperative_ctde_validation.json
  logs/
  happo/E1_seed_0/
  happo/E2_seed_0/
  happo/E3_seed_0/
  masac/E1_seed_0/
  ...
```

Cada corrida debe conservar `results.json`, `training_summary.json`, `timeseries.csv`, `trace.csv`, `checkpoint_manifest.json`, `checkpoints/`, `figures/` y `figures/tables/`.


## Perfil GPU local y limite real de MASAC

El lanzamiento oficial vigente usa un perfil **GPU-tuned conservador** para la RTX 4060 Laptop de 8 GB: redes de 384 unidades en HAPPO/MATD3/MAAC, lotes `512` en MATD3/MAAC, `live_progress_interval=250`, y MASAC con `buffer_size=8`, `critic_batch_size=2`, `critic_train_steps=2`, `actor_sample_times=8`, `rnn_hidden_dim=128`, `qmix_hidden_dim=64` e `hyper_hidden_dim=128`.

En MASAC puede verse memoria GPU alta y utilizacion baja. Esto no significa que CUDA este fallando: el backend oficial alterna entre simulacion secuencial del entorno CityLearn para 17 edificios + EV y actualizaciones PyTorch. Durante el rollout, el cuello de botella es CPU/Python/CityLearn; la GPU se activa mas durante las actualizaciones de red. Subir mas los lotes en la GPU local no necesariamente acelera, porque con 8 GB de VRAM aumenta el costo por actualizacion y el riesgo de quedarse sin memoria.


In [ ]:
# Configura o actualiza `OFFICIAL_OUTPUT`.
OFFICIAL_OUTPUT = PROJECT_ROOT / 'outputs' / 'citylearn_v3_madrl_official_full_cuda_v2'
# Construye o ejecuta comandos externos del flujo de trabajo.
OFFICIAL_COMMAND = [
    # Declara un argumento o componente del comando ejecutable.
    'powershell.exe',
    # Ejecuta una instruccion necesaria para esta celda.
    '-NoProfile',
    # Ejecuta una instruccion necesaria para esta celda.
    '-ExecutionPolicy', 'Bypass',
    # Ejecuta una instruccion necesaria para esta celda.
    '-File', str(SCRIPTS_DIR / 'launch_citylearn_v3_official_training.ps1'),
    # Ejecuta una instruccion necesaria para esta celda.
    '-Scenario', OFFICIAL_SCENARIO,
    # Ejecuta una instruccion necesaria para esta celda.
    '-Seed', str(RANDOM_SEED),
    # Ejecuta una instruccion necesaria para esta celda.
    '-EpisodeTimeSteps', str(OFFICIAL_EPISODE_TIME_STEPS),
    # Ejecuta una instruccion necesaria para esta celda.
    '-Episodes', '5',
    # Ejecuta una instruccion necesaria para esta celda.
    '-OutputRoot', str(OFFICIAL_OUTPUT),
    # Ejecuta una instruccion necesaria para esta celda.
    '-TorchThreads', '12',
    # Ejecuta una instruccion necesaria para esta celda.
    '-Cuda',
# Cierra la estructura de datos o llamada definida arriba.
]
# Muestra informacion de seguimiento para el usuario.
print(' '.join(map(str, OFFICIAL_COMMAND)))

In [ ]:
# Configura o actualiza `RUN_OFFICIAL_TRAINING`.
RUN_OFFICIAL_TRAINING = False

# Evalua una condicion antes de continuar el flujo.
if RUN_OFFICIAL_TRAINING:
    # Construye o ejecuta comandos externos del flujo de trabajo.
    subprocess.Popen(OFFICIAL_COMMAND, cwd=PROJECT_ROOT)
    # Muestra informacion de seguimiento para el usuario.
    print('Official training launched in background.')
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('Set RUN_OFFICIAL_TRAINING=True only when you intend to launch the long official run.')

## Monitor visual local

El proyecto incluye un monitor PowerShell que muestra la matriz `E1/E2/E3 x HAPPO/MASAC/MATD3/MAAC`, uso GPU, proceso activo, `global_step`, episodio, `episode_step`, recompensa instantanea, retorno acumulado, funcion reward, perfil MADRL, pesos activos, costo, CO2, carga neta y artefactos recientes.

En las salidas actuales, `reward_sum` y `reward_mean` se conservan por compatibilidad pero representan la recompensa instantanea del paso (`instant_step_sum` e `instant_step_mean`). Para analizar aprendizaje y convergencia se deben usar `episode_return_cumulative`, `total_return_cumulative`, `convergence_returns.png`, `episode_reward_summary.png` y `learning_efficiency.png`.


In [ ]:
# Construye o ejecuta comandos externos del flujo de trabajo.
MONITOR_COMMAND = [
    # Declara un argumento o componente del comando ejecutable.
    'powershell.exe',
    # Ejecuta una instruccion necesaria para esta celda.
    '-NoProfile',
    # Ejecuta una instruccion necesaria para esta celda.
    '-ExecutionPolicy', 'Bypass',
    # Ejecuta una instruccion necesaria para esta celda.
    '-File', str(SCRIPTS_DIR / 'monitor_citylearn_v3_official_training.ps1'),
    # Ejecuta una instruccion necesaria para esta celda.
    '-OutputRoot', str(OFFICIAL_OUTPUT),
    # Ejecuta una instruccion necesaria para esta celda.
    '-IntervalSeconds', '5',
    # Ejecuta una instruccion necesaria para esta celda.
    '-LogTail', '20',
# Cierra la estructura de datos o llamada definida arriba.
]
# Muestra informacion de seguimiento para el usuario.
print(' '.join(map(str, MONITOR_COMMAND)))

## Google Colab GPU A100/T4 Training Cell

Esta celda permite ejecutar el entrenamiento en Google Colab usando GPU. Esta pensada para Colab Pro/Pro+ con **A100** cuando este disponible; tambien funciona con T4/V100, pero sera mas lento. En Colab Linux no se usa el launcher PowerShell; por eso la celda ejecuta directamente los cuatro scripts `train_citylearn_v3_*.py` para cada escenario `E1`, `E2` y `E3`.

La celda esta apagada por defecto. Cambia `RUN_COLAB_GPU_TRAINING = True` solo cuando el repositorio ya este clonado con submodulos y las dependencias instaladas.


In [ ]:
# Configura o actualiza `RUN_COLAB_GPU_TRAINING`.
RUN_COLAB_GPU_TRAINING = False
# Configura o actualiza `COLAB_OUTPUT`.
COLAB_OUTPUT = PROJECT_ROOT / 'outputs' / 'citylearn_v3_madrl_colab_gpu'
# Configura o actualiza `COLAB_EPISODES`.
COLAB_EPISODES = 5
# Configura o actualiza `COLAB_EPISODE_TIME_STEPS`.
COLAB_EPISODE_TIME_STEPS = 8760
# Configura o actualiza `COLAB_SCENARIOS`.
COLAB_SCENARIOS = ['E1', 'E2', 'E3']
# Configura o actualiza `COLAB_ALGORITHMS`.
COLAB_ALGORITHMS = ['happo', 'masac', 'matd3', 'maac']


# Define la funcion auxiliar `is_google_colab`.
def is_google_colab() -> bool:
    # Inicia un bloque protegido para capturar errores controlados.
    try:
        # Importa dependencias necesarias para esta seccion.
        import google.colab  # type: ignore
        # Retorna el resultado calculado por la funcion.
        return True
    # Captura una excepcion para manejarla sin detener todo el notebook.
    except Exception:
        # Retorna el resultado calculado por la funcion.
        return False


# Define la funcion auxiliar `colab_madrl_command`.
def colab_madrl_command(algorithm: str, scenario: str) -> List[str]:
    # Configura o actualiza `script`.
    script = SCRIPTS_DIR / f'train_citylearn_v3_{algorithm}.py'
    # Configura o actualiza `num_env_steps`.
    num_env_steps = COLAB_EPISODE_TIME_STEPS * COLAB_EPISODES
    # Construye o ejecuta comandos externos del flujo de trabajo.
    command = [
        # Ejecuta una instruccion necesaria para esta celda.
        sys.executable,
        # Ejecuta una instruccion necesaria para esta celda.
        '-B',
        # Ejecuta una instruccion necesaria para esta celda.
        str(script),
        # Declara un argumento o componente del comando ejecutable.
        '--scenario', scenario,
        # Declara un argumento o componente del comando ejecutable.
        '--seed', str(RANDOM_SEED),
        # Declara un argumento o componente del comando ejecutable.
        '--episode-time-steps', str(COLAB_EPISODE_TIME_STEPS),
        # Declara un argumento o componente del comando ejecutable.
        '--episodes', str(COLAB_EPISODES),
        # Declara un argumento o componente del comando ejecutable.
        '--output-dir', str(COLAB_OUTPUT / algorithm),
        # Declara un argumento o componente del comando ejecutable.
        '--cuda',
    # Cierra la estructura de datos o llamada definida arriba.
    ]

    # Evalua una condicion antes de continuar el flujo.
    if algorithm == 'happo':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--num-env-steps', str(num_env_steps),
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--torch-threads', '12',
            # Declara un argumento o componente del comando ejecutable.
            '--n-rollout-threads', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--log-interval', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--eval-interval', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'masac':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--action-bins', '3',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-size', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--critic-batch-size', '2',
            # Declara un argumento o componente del comando ejecutable.
            '--critic-train-steps', '2',
            # Declara un argumento o componente del comando ejecutable.
            '--actor-sample-times', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--rnn-hidden-dim', '128',
            # Declara un argumento o componente del comando ejecutable.
            '--qmix-hidden-dim', '64',
            # Declara un argumento o componente del comando ejecutable.
            '--hyper-hidden-dim', '128',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'matd3':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--num-env-steps', str(num_env_steps),
            # Declara un argumento o componente del comando ejecutable.
            '--batch-size', '512',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-size', '50000',
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--train-interval', '100',
            # Declara un argumento o componente del comando ejecutable.
            '--num-random-episodes', '1',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Evalua una condicion alternativa.
    elif algorithm == 'maac':
        # Ejecuta una instruccion necesaria para esta celda.
        command.extend([
            # Declara un argumento o componente del comando ejecutable.
            '--action-bins', '3',
            # Declara un argumento o componente del comando ejecutable.
            '--batch-size', '512',
            # Declara un argumento o componente del comando ejecutable.
            '--buffer-length', '200000',
            # Declara un argumento o componente del comando ejecutable.
            '--steps-per-update', '250',
            # Declara un argumento o componente del comando ejecutable.
            '--num-updates', '8',
            # Declara un argumento o componente del comando ejecutable.
            '--hidden-size', '384',
            # Declara un argumento o componente del comando ejecutable.
            '--attend-heads', '4',
            # Declara un argumento o componente del comando ejecutable.
            '--pi-lr', '0.0003',
            # Declara un argumento o componente del comando ejecutable.
            '--q-lr', '0.001',
            # Declara un argumento o componente del comando ejecutable.
            '--tau', '0.005',
            # Declara un argumento o componente del comando ejecutable.
            '--gamma', '0.99',
            # Declara un argumento o componente del comando ejecutable.
            '--live-progress-interval', '250',
        # Cierra la estructura de datos o llamada definida arriba.
        ])
    # Ejecuta la rama alternativa cuando la condicion previa no se cumple.
    else:
        # Interrumpe la ejecucion con un error explicito si falla una condicion critica.
        raise ValueError(f'Unknown algorithm: {algorithm}')

    # Retorna el resultado calculado por la funcion.
    return command


# Evalua una condicion antes de continuar el flujo.
if RUN_COLAB_GPU_TRAINING:
    # Importa dependencias necesarias para esta seccion.
    import torch

    # Evalua una condicion antes de continuar el flujo.
    if not is_google_colab():
        # Muestra informacion de seguimiento para el usuario.
        print('Aviso: esta celda tambien puede correr localmente, pero fue preparada para Google Colab.')

    # Evalua una condicion antes de continuar el flujo.
    if not torch.cuda.is_available():
        # Interrumpe la ejecucion con un error explicito si falla una condicion critica.
        raise RuntimeError('CUDA no esta disponible. En Colab ve a Runtime > Change runtime type > GPU.')

    # Configura o actualiza `device_name`.
    device_name = torch.cuda.get_device_name(0)
    # Muestra informacion de seguimiento para el usuario.
    print('CUDA device:', device_name)
    # Evalua una condicion antes de continuar el flujo.
    if 'A100' not in device_name.upper():
        # Muestra informacion de seguimiento para el usuario.
        print('Aviso: no parece ser A100. El entrenamiento puede funcionar, pero sera mas lento.')

    # Itera sobre una coleccion de elementos.
    for scenario in COLAB_SCENARIOS:
        # Itera sobre una coleccion de elementos.
        for algorithm in COLAB_ALGORITHMS:
            # Construye o ejecuta comandos externos del flujo de trabajo.
            command = colab_madrl_command(algorithm, scenario)
            # Muestra informacion de seguimiento para el usuario.
            print('\n===', algorithm.upper(), scenario, '===')
            # Muestra informacion de seguimiento para el usuario.
            print(' '.join(map(str, command)))
            # Construye o ejecuta comandos externos del flujo de trabajo.
            subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    # Muestra informacion de seguimiento para el usuario.
    print('Colab GPU training completed:', COLAB_OUTPUT)
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('Set RUN_COLAB_GPU_TRAINING=True only in Google Colab with GPU enabled.')
    # Muestra informacion de seguimiento para el usuario.
    print('Recommended runtime: Colab Pro/Pro+ A100. T4/V100 also works with longer runtime.')

# Evaluate the Episode Rewards for MADRL Algorithms

Las recompensas no sustituyen los KPIs. Sirven para diagnosticar aprendizaje, convergencia, exploracion y estabilidad. Una politica con recompensa creciente aun puede fallar en CO2 o costo si la recompensa no captura bien esos objetivos.


In [ ]:
# Define la funcion auxiliar `available_run_dirs`.
def available_run_dirs(output_root: Path, scenarios: Sequence[str] = SCENARIOS) -> Dict[str, Path]:
    # Configura o actualiza `Path]`.
    runs: Dict[str, Path] = {}
    # Itera sobre una coleccion de elementos.
    for algorithm in ALGORITHMS:
        # Itera sobre una coleccion de elementos.
        for scenario in scenarios:
            # Configura o actualiza `path`.
            path = run_dir(output_root, algorithm, scenario=scenario)
            # Evalua una condicion antes de continuar el flujo.
            if path.exists():
                # Configura o actualiza `runs[f'{algorithm}_{scenario}']`.
                runs[f'{algorithm}_{scenario}'] = path
    # Retorna el resultado calculado por la funcion.
    return runs


# Configura o actualiza `runs`.
runs = available_run_dirs(OFFICIAL_OUTPUT)
# Muestra informacion de seguimiento para el usuario.
print('Available runs:')
# Itera sobre una coleccion de elementos.
for run_name, path in runs.items():
    # Muestra informacion de seguimiento para el usuario.
    print(run_name, '->', path)

In [ ]:
# Configura o actualiza `reward_rows`.
reward_rows = []
# Itera sobre una coleccion de elementos.
for run_name, path in runs.items():
    # Configura o actualiza `timeseries`.
    timeseries = load_run_timeseries(path)
    # Evalua una condicion antes de continuar el flujo.
    if timeseries.empty or 'reward_sum' not in timeseries.columns:
        # Ejecuta una instruccion necesaria para esta celda.
        continue
    # Configura o actualiza `grouped`.
    grouped = timeseries.groupby('episode', as_index=False).agg(
        # Configura o actualiza `reward_total`.
        reward_total=('reward_sum', 'sum'),
        # Configura o actualiza `reward_mean`.
        reward_mean=('reward_mean', 'mean'),
        # Configura o actualiza `steps`.
        steps=('global_step', 'count'),
    # Cierra la estructura de datos o llamada definida arriba.
    )
    # Configura o actualiza `grouped['run']`.
    grouped['run'] = run_name.upper()
    # Ejecuta una instruccion necesaria para esta celda.
    reward_rows.append(grouped)

# Evalua una condicion antes de continuar el flujo.
if reward_rows:
    # Configura o actualiza `reward_table`.
    reward_table = pd.concat(reward_rows, ignore_index=True)
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(reward_table)
    # Configura o dibuja una visualizacion.
    fig, ax = plt.subplots(figsize=(10, 4))
    # Itera sobre una coleccion de elementos.
    for run_name, group in reward_table.groupby('run'):
        # Configura o dibuja una visualizacion.
        ax.plot(group['episode'], group['reward_total'], marker='o', label=run_name)
    # Configura o dibuja una visualizacion.
    ax.set_title('Episode returns by MADRL run')
    # Configura o dibuja una visualizacion.
    ax.set_xlabel('episode')
    # Configura o dibuja una visualizacion.
    ax.set_ylabel('reward_total')
    # Configura o dibuja una visualizacion.
    ax.legend()
    # Configura o dibuja una visualizacion.
    plt.show()
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('No reward tables available yet.')

# Analyze Training Artifacts

Cada MADRL produce `data/results.json`, `data/timeseries.csv`, `data/trace.csv`, `data/checkpoint_manifest.json` y una carpeta `figures/` con graficas y tablas.


In [ ]:
# Configura o actualiza `REGENERATE_FIGURES`.
REGENERATE_FIGURES = False

# Evalua una condicion antes de continuar el flujo.
if REGENERATE_FIGURES:
    # Itera sobre una coleccion de elementos.
    for run_name, path in runs.items():
        # Construye o ejecuta comandos externos del flujo de trabajo.
        command = [sys.executable, '-B', str(SCRIPTS_DIR / 'regenerate_citylearn_v3_figures.py'), str(path)]
        # Muestra informacion de seguimiento para el usuario.
        print('Regenerating figures for', run_name.upper())
        # Construye o ejecuta comandos externos del flujo de trabajo.
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('Set REGENERATE_FIGURES=True to rebuild figures/tables from saved CSV/JSON artifacts.')

Las figuras obligatorias cubren rendimiento, eficiencia, convergencia, comparacion con baseline, evolucion de entrenamiento, exploracion, aprendizaje, recompensas, returns, ganancias y perfiles por eje.


In [ ]:
# Evalua una condicion antes de continuar el flujo.
if runs:
    # Configura o actualiza `first_path`.
    first_run_name, first_path = next(iter(runs.items()))
    # Muestra informacion de seguimiento para el usuario.
    print('Showing figures for:', first_run_name.upper())
    # Configura o actualiza `names`.
    display_generated_figures(first_path, names=[
        # Ejecuta una instruccion necesaria para esta celda.
        'reward_timeseries.png',
        # Ejecuta una instruccion necesaria para esta celda.
        'convergence_returns.png',
        # Ejecuta una instruccion necesaria para esta celda.
        'learning_efficiency.png',
        # Ejecuta una instruccion necesaria para esta celda.
        'citylearn_v2_district_timeseries.png',
        # Ejecuta una instruccion necesaria para esta celda.
        'exploration_action_l2.png',
        # Ejecuta una instruccion necesaria para esta celda.
        'baseline_gain_by_kpi.png',
    # Cierra la estructura de datos o llamada definida arriba.
    ])
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('No run directories found yet. Train or point OFFICIAL_OUTPUT to an existing run root.')

# Compare MADRL Algorithms Against Baseline

La comparacion debe hacerse por eje. No se recomienda mezclar todos los KPIs en un unico numero sin explicar pesos y normalizacion. Primero miramos KPIs por algoritmo; despues se puede aplicar TOPSIS o ranking ponderado.


In [ ]:
# Configura o actualiza `all_kpi_tables`.
all_kpi_tables = []
# Itera sobre una coleccion de elementos.
for run_name, path in runs.items():
    # Configura o actualiza `table`.
    table = load_objective_kpis(path)
    # Evalua una condicion antes de continuar el flujo.
    if not table.empty:
        # Configura o actualiza `table['run']`.
        table['run'] = run_name.upper()
        # Configura o actualiza `table['algorithm']`.
        table['algorithm'] = run_name.split('_')[0].upper()
        # Configura o actualiza `table['scenario']`.
        table['scenario'] = run_name.split('_')[1].upper() if '_' in run_name else SCENARIO
        # Ejecuta una instruccion necesaria para esta celda.
        all_kpi_tables.append(table)

# Evalua una condicion antes de continuar el flujo.
if all_kpi_tables:
    # Configura o actualiza `kpi_comparison`.
    kpi_comparison = pd.concat(all_kpi_tables, ignore_index=True)
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(kpi_comparison.head(20))
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Configura o actualiza `kpi_comparison`.
    kpi_comparison = pd.DataFrame()
    # Muestra informacion de seguimiento para el usuario.
    print('No objective_kpis.csv tables available yet.')

In [ ]:
# Define la funcion auxiliar `plot_algorithm_kpi_comparison`.
def plot_algorithm_kpi_comparison(kpi_table: pd.DataFrame, axis: str, kpis: Sequence[str]) -> Optional[plt.Figure]:
    # Evalua una condicion antes de continuar el flujo.
    if kpi_table.empty:
        # Retorna el resultado calculado por la funcion.
        return None
    # Configura o actualiza `subset`.
    subset = kpi_table[(kpi_table['axis'] == axis) & (kpi_table['kpi'].isin(kpis))].copy()
    # Evalua una condicion antes de continuar el flujo.
    if subset.empty:
        # Retorna el resultado calculado por la funcion.
        return None
    # Configura o actualiza `pivot`.
    pivot = subset.pivot_table(index='kpi', columns='algorithm', values='value', aggfunc='first')
    # Configura o dibuja una visualizacion.
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(pivot))))
    # Configura o actualiza `pivot.plot(kind`.
    pivot.plot(kind='barh', ax=ax)
    # Configura o dibuja una visualizacion.
    ax.set_title(f'{axis} algorithm KPI comparison')
    # Configura o dibuja una visualizacion.
    ax.set_xlabel('value')
    # Ejecuta una instruccion necesaria para esta celda.
    fig.tight_layout()
    # Retorna el resultado calculado por la funcion.
    return fig

# Evalua una condicion antes de continuar el flujo.
if not kpi_comparison.empty:
    # Configura o actualiza `manifest`.
    manifest = objective_manifest()
    # Itera sobre una coleccion de elementos.
    for axis, payload in manifest['axes'].items():
        # Configura o dibuja una visualizacion.
        fig = plot_algorithm_kpi_comparison(kpi_comparison, axis, payload['kpis'])
        # Evalua una condicion antes de continuar el flujo.
        if fig is None:
            # Muestra informacion de seguimiento para el usuario.
            print(f'No KPI values available for {axis}.')
            # Ejecuta una instruccion necesaria para esta celda.
            continue
        # Configura o dibuja una visualizacion.
        plt.show()

# Tune your MADRL Experiment

Como en el tutorial original se ajusta SAC, aqui se ajusta la configuracion experimental MADRL: horizonte de entrenamiento, semillas, escenario E1/E2/E3, pesos multiobjetivo, arquitectura, frecuencia de actualizacion, tamano de buffer, tasas de aprendizaje, discretizacion de acciones, cabezas de atencion y retardo de politica.


In [ ]:
# Importa dependencias necesarias para esta seccion.
from citylearn.v3.config import CityLearnV3ExperimentConfig

# Configura o actualiza `config`.
config = CityLearnV3ExperimentConfig()
# Muestra informacion de seguimiento para el usuario.
print('Algorithms:', config.algorithms)
# Muestra informacion de seguimiento para el usuario.
print('Scenarios:', config.scenarios)
# Muestra informacion de seguimiento para el usuario.
print('Seeds:', config.seeds)
# Muestra informacion de seguimiento para el usuario.
print('Episode time steps:', config.episode_time_steps)
# Muestra informacion de seguimiento para el usuario.
print('Reward function:', config.reward_function)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(pd.Series(config.hyperparameters).to_frame('value'))
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(axis_reward_table)
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(reward_profile_table)

In [ ]:
# Configura o actualiza `official_hyperparameters`.
official_hyperparameters = pd.DataFrame([
    # Ejecuta una instruccion necesaria para esta celda.
    {
        # Ejecuta una instruccion necesaria para esta celda.
        'algorithm': 'HAPPO',
        # Configura o actualiza `'hidden_size`.
        'main_parameters': 'hidden_size=256, share_param=False, n_rollout_threads=1',
        # Configura o actualiza `'episode_length`.
        'training': 'episode_length=8760, num_env_steps=43800, log_interval=1, eval_interval=1',
        # Configura o actualiza `lr`.
        'optimizer': 'HARL official defaults: lr=5e-4, critic_lr=5e-4, gamma=0.99, gae_lambda=0.95, clip=0.2',
    # Cierra la estructura de datos o llamada definida arriba.
    },
    # Ejecuta una instruccion necesaria para esta celda.
    {
        # Ejecuta una instruccion necesaria para esta celda.
        'algorithm': 'MASAC',
        # Configura o actualiza `'action_bins`.
        'main_parameters': 'action_bins=3, critic_batch_size=1, buffer_size=2',
        # Configura o actualiza `'episodes`.
        'training': 'episodes=5, n_epoch=5, n_episodes=1, episode_limit=8760',
        # Ejecuta una instruccion necesaria para esta celda.
        'optimizer': 'paper backend defaults plus CityLearn v3 CTDE state_shape',
    # Cierra la estructura de datos o llamada definida arriba.
    },
    # Ejecuta una instruccion necesaria para esta celda.
    {
        # Ejecuta una instruccion necesaria para esta celda.
        'algorithm': 'MATD3',
        # Configura o actualiza `'batch_size`.
        'main_parameters': 'batch_size=256, buffer_size=10000, hidden_size=256',
        # Configura o actualiza `'num_env_steps`.
        'training': 'num_env_steps=43800, train_interval=100, num_random_episodes=1',
        # Configura o actualiza `lr`.
        'optimizer': 'off-policy defaults: lr=5e-4, gamma=0.99, tau=0.005, target_noise=0.2',
    # Cierra la estructura de datos o llamada definida arriba.
    },
    # Ejecuta una instruccion necesaria para esta celda.
    {
        # Ejecuta una instruccion necesaria para esta celda.
        'algorithm': 'MAAC',
        # Configura o actualiza `'batch_size`.
        'main_parameters': 'batch_size=256, buffer_length=100000, hidden_size=256, attend_heads=4',
        # Configura o actualiza `'steps_per_update`.
        'training': 'steps_per_update=100, num_updates=4, episodes=5',
        # Configura o actualiza `'pi_lr`.
        'optimizer': 'pi_lr=3e-4, q_lr=1e-3, tau=0.005, gamma=0.99, reward_scale=100',
    # Cierra la estructura de datos o llamada definida arriba.
    },
# Cierra la estructura de datos o llamada definida arriba.
])
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(official_hyperparameters)

## Set Environment, Agent and Reward Function

Para resultados de tesis no basta una corrida. Una matriz minima defendible usa los cuatro algoritmos, tres escenarios, varias semillas, mismo dataset, mismo horizonte, mismos KPIs y comparacion contra baseline CityLearn v2.


In [ ]:
# Configura o actualiza `experiment_matrix`.
experiment_matrix = pd.MultiIndex.from_product(
    # Ejecuta una instruccion necesaria para esta celda.
    [config.algorithms, config.scenarios, config.seeds[:3]],
    # Configura o actualiza `names`.
    names=['algorithm', 'scenario', 'seed'],
# Cierra la estructura de datos o llamada definida arriba.
).to_frame(index=False)
# Muestra informacion de seguimiento para el usuario.
print('Example experiment count with first 3 seeds:', len(experiment_matrix))
# Renderiza una tabla, figura o Markdown dentro del notebook.
display(experiment_matrix.head(20))

## Submit

En lugar de una celda de envio a leaderboard, este proyecto usa trazabilidad local/GitHub: codigo, `backends.lock.json`, `official_full_manifest.json`, `official_full_status.json`, checkpoints, JSON/CSV/PNG/MD por corrida.


In [ ]:
# Configura o actualiza `status_path`.
status_path = OFFICIAL_OUTPUT / 'official_full_status.json'
# Configura o actualiza `manifest_path`.
manifest_path = OFFICIAL_OUTPUT / 'official_full_manifest.json'
# Configura o actualiza `cooperative_validation_path`.
cooperative_validation_path = OFFICIAL_OUTPUT / 'cooperative_ctde_validation.json'

# Evalua una condicion antes de continuar el flujo.
if status_path.is_file():
    # Configura o actualiza `status`.
    status = load_json(status_path)
    # Configura o actualiza `reward_status`.
    reward_status = status.get('reward', {}) if isinstance(status.get('reward'), Mapping) else {}
    # Renderiza una tabla, figura o Markdown dentro del notebook.
    display(pd.Series({
        # Ejecuta una instruccion necesaria para esta celda.
        'status': status.get('status'),
        # Ejecuta una instruccion necesaria para esta celda.
        'dataset': status.get('dataset'),
        # Ejecuta una instruccion necesaria para esta celda.
        'scenario': status.get('scenario'),
        # Ejecuta una instruccion necesaria para esta celda.
        'scenarios': ', '.join(status.get('scenarios', [])) if isinstance(status.get('scenarios'), list) else status.get('scenarios'),
        # Ejecuta una instruccion necesaria para esta celda.
        'episodes': status.get('episodes'),
        # Ejecuta una instruccion necesaria para esta celda.
        'episode_time_steps': status.get('episode_time_steps'),
        # Ejecuta una instruccion necesaria para esta celda.
        'num_env_steps': status.get('num_env_steps'),
        # Ejecuta una instruccion necesaria para esta celda.
        'torch': status.get('torch'),
        # Ejecuta una instruccion necesaria para esta celda.
        'cuda': status.get('cuda'),
        # Ejecuta una instruccion necesaria para esta celda.
        'reward_function': reward_status.get('function'),
        # Ejecuta una instruccion necesaria para esta celda.
        'reward_aggregation': reward_status.get('aggregation'),
        # Ejecuta una instruccion necesaria para esta celda.
        'not_using_marl_base_weights': reward_status.get('not_using_marl_base_weights'),
        # Ejecuta una instruccion necesaria para esta celda.
        'output_root': status.get('output_root'),
    # Cierra la estructura de datos o llamada definida arriba.
    }).to_frame('value'))

    # Configura o actualiza `jobs`.
    jobs = status.get('jobs', [])
    # Evalua una condicion antes de continuar el flujo.
    if jobs:
        # Renderiza una tabla, figura o Markdown dentro del notebook.
        display(pd.DataFrame(jobs)[['scenario', 'name', 'started_at', 'completed_at', 'exit_code', 'output_dir']])
# Ejecuta la rama alternativa cuando la condicion previa no se cumple.
else:
    # Muestra informacion de seguimiento para el usuario.
    print('No official status file found at', status_path)

# Evalua si existe la validacion cooperativa CTDE.
if cooperative_validation_path.is_file():
    # Carga la validacion cooperativa CTDE.
    cooperative_status = load_json(cooperative_validation_path)
    # Muestra el estado de validacion cooperativa.
    print('cooperative_ctde_validation_status:', cooperative_status.get('status'))
    # Convierte las filas de validacion a tabla.
    cooperative_rows = pd.DataFrame(cooperative_status.get('rows', []))
    # Evalua si existen filas de validacion para mostrar.
    if not cooperative_rows.empty:
        # Renderiza una vista resumida por algoritmo y escenario.
        display(cooperative_rows[['algorithm', 'scenario', 'agents', 'reward_aggregation', 'state_dim', 'obs_dim_sum', 'not_using_marl_base_weights']])
# Ejecuta la rama alternativa cuando no existe validacion cooperativa.
else:
    # Muestra la ruta esperada para la validacion cooperativa.
    print('No cooperative CTDE validation file found at', cooperative_validation_path)


# Referencias seleccionadas en formato APA 7

Ackermann, J., Gabler, V., Osa, T., & Sugiyama, M. (2019). *Reducing overestimation bias in multi-agent domains using double centralized critics*. arXiv. https://doi.org/10.48550/arXiv.1910.01465

Almannouny, G. A. (2025). *Intelligent dynamic pricing and integrated demand response for multi-energy systems using deep reinforcement learning* [Doctoral dissertation, University of Glasgow]. Enlighten Theses. https://doi.org/10.5525/gla.thesis.85367

Bernstein, D. S., Givan, R., Immerman, N., & Zilberstein, S. (2002). The complexity of decentralized control of Markov decision processes. *Mathematics of Operations Research, 27*(4), 819-840. https://doi.org/10.1287/moor.27.4.819.297

Dong, J. (2022). *Peak load ensemble prediction and multi-agent reinforcement learning for DER demand response management in smart grids* [Master's thesis, Lakehead University]. Knowledge Commons. https://knowledgecommons.lakeheadu.ca/handle/2453/4944

Fonseca, T. C. C. (2023). *A multi-agent reinforcement learning approach to integrate flexible assets into energy communities* [Master's thesis, Instituto Superior de Engenharia do Porto]. Repositório Científico do Instituto Politécnico do Porto. http://hdl.handle.net/10400.22/24068

González Rotger, C. (2021). *Multi-agent reinforcement learning applied to heating, ventilation, and air conditioning in a building energy management system* [Master's thesis, Universitat de les Illes Balears]. http://hdl.handle.net/11201/158415

Hu, S., Zhong, Y., Gao, M., Wang, W., Dong, H., Liang, X., Li, Z., Chang, X., & Yang, Y. (2023). *MARLlib: A scalable and efficient multi-agent reinforcement learning library*. arXiv. https://arxiv.org/abs/2210.13708

International Energy Agency. (2023). *Buildings*. https://www.iea.org/energy-system/buildings

Iqbal, S., & Sha, F. (2019). *Actor-attention-critic for multi-agent reinforcement learning*. arXiv. https://doi.org/10.48550/arXiv.1810.02912

Lowe, R., Wu, Y., Tamar, A., Harb, J., Abbeel, P., & Mordatch, I. (2017). Multi-agent actor-critic for mixed cooperative-competitive environments. *Advances in Neural Information Processing Systems, 30*. https://papers.nips.cc/paper/7217-multi-agent-actor-critic-for-mixed-cooperative-competitive-environments

Nweye, K., Kaspar, K., Buscemi, G., Fonseca, T., Pinto, G., Ghose, D., Duddukuru, S., Pratapa, P., Li, H., Mohammadi, J., Lino Ferreira, L., Hong, T., Ouf, M., Capozzoli, A., & Nagy, Z. (2025). CityLearn v2: Energy-flexible, resilient, occupant-centric, and carbon-aware management of grid-interactive communities. *Journal of Building Performance Simulation, 18*(1), 17-38. https://doi.org/10.1080/19401493.2024.2418813

Pu, Y., Wang, S., Yang, R., Yao, X., & Li, B. (2021). *Decomposed soft actor-critic method for cooperative multi-agent reinforcement learning*. arXiv. https://doi.org/10.48550/arXiv.2104.06655

Rashid, T., Samvelyan, M., Schroeder de Witt, C., Farquhar, G., Foerster, J., & Whiteson, S. (2018). *QMIX: Monotonic value function factorisation for deep multi-agent reinforcement learning*. arXiv. https://doi.org/10.48550/arXiv.1803.11485

United Nations Environment Programme & Global Alliance for Buildings and Construction. (2024). *Global status report for buildings and construction*. https://www.unep.org/resources/report/global-status-report-buildings-and-construction

Vázquez-Canteli, J. R., Dey, S., Henze, G., & Nagy, Z. (2020). *CityLearn: Standardizing research in multi-agent reinforcement learning for demand response and urban energy management*. arXiv. https://doi.org/10.48550/arXiv.2012.10504

Zhong, Y., Kuba, J. G., Feng, X., Hu, S., Ji, J., & Yang, Y. (2024). Heterogeneous-agent reinforcement learning. *Journal of Machine Learning Research, 25*(32), 1-67. https://jmlr.org/papers/v25/23-0488.html


# Next Steps

1. Reiniciar el entrenamiento oficial secuencial despues de limpiar salidas creadas con rewards anteriores.
2. Regenerar figuras si alguna corrida fue creada antes de `CityLearnV3MADRLRewardFunction`.
3. Consolidar `objective_kpis.csv` de HAPPO, MASAC, MATD3 y MAAC.
4. Crear tablas comparativas por OE1/OE2/OE3.
5. Aplicar ranking TOPSIS o ponderado con pesos justificados.
6. Revisar estabilidad por semilla y no solo una corrida.
7. Reportar limitaciones: costo computacional, discretizacion de algunos backends y dependencia de calidad de recompensa.


## Other Ideas

- Ejecutar E1, E2 y E3 por separado para observar especializacion de politicas.
- Comparar `team_mean`, `team_sum`, `individual` y recompensa mixta.
- Incorporar analisis de equidad entre edificios.
- Evaluar sensibilidad a EV penetration y disponibilidad PV.
- Generar curvas de Pareto entre flexibilidad, CO2 y costo.
- Usar MARLlib para experimentos adicionales con el mismo adaptador `citylearn_v3`.
